# TourFold v3 · Gezgin Satıcı Problemi (TSP)

**Tek hücre, tam otomatik; yeni eğitim yok.** v2'nin Drive'daki TourFold-v2 modelini ve ilk çalıştırmanın verisini (`MyDrive/TourFold_TSP`) yeniden kullanır. Hücreyi çalıştırın ve Drive izin penceresinde onay verin.

- **Soru:** TourFold'un adayları LKH'yi hızlandırıyor mu? Aynı örnekler ve süre bütçeleriyle 8 yapılandırma: LKH-3 POPMUSIC, kontrol (POPMUSIC adayları + topluluğun en iyi turu), sayım+, TourFold-v2 ilk 5, karma listeler (TourFold 3 + POPMUSIC 2, TourFold 3 + en yakın 2, TourFold ∪ POPMUSIC) ve 4 turluk topluluk.
- **Süre:** yaklaşık 2,5 saat (L4, 12 vCPU). Tarayıcı sekmesini açık bırakın; aynı anda başka bir TourFold defteri çalıştırmayın (süre ölçümleri bozulur).
- **Sonuçlar:** `MyDrive/TourFold_TSP/v3` — rapor `RAPOR_v3.md`, ilerleme ve kalan süre `log_v3.txt` / `status_v3.json`. Bitince `TourFold_v3_sonuclar.zip` Drive'a kaydedilir ve tarayıcınıza iner.
- **Bağlantı koparsa:** hücreyi yeniden çalıştırın; biten adımlar ve tamamlanan LKH parçaları atlanır.
- **Donanım:** L4 GPU ve yüksek RAM.


In [ ]:
# ==== TourFold TSP v3 · tek hücre, tam otomatik =================================================
# v3 yeni eğitim yapmaz: v2'nin Drive'daki TourFold-v2 modelini ve ilk çalıştırmanın verisini yeniden kullanır.
# Soru: TourFold'un adayları LKH'yi nasıl hızlandırır? Aynı örnekler ve aynı süre bütçeleriyle 8 yapılandırma:
#   * LKH-3 POPMUSIC (taban) ve kontrol: POPMUSIC adayları + topluluğun en iyi turu başlangıç olarak
#     (başlangıç turunun etkisini adayların etkisinden ayırır)
#   * sayım+ (topluluk sıklığı) ve TourFold-v2'nin ilk 5 adayı (v2'nin tekrarı)
#   * karma listeler: TourFold'un ilk 3 adayı + POPMUSIC'in ya da en yakın komşuların 2 adayı; TourFold ∪ POPMUSIC
#   * 4 turluk topluluk (topluluk maliyetinin bir kısmı kalkar)
# Sırası: LKH kurulumu ve sınaması -> kayıtlı verinin ve TourFold-v2 modelinin yüklenmesi -> aday listesi kalitesi ->
# LKH karşılaştırması -> RAPOR_v3.md ve sonuç paketi (Drive'a kaydedilir, tarayıcıya iner).
# Elle yapılacak tek şey: Google Drive izni. Sonuçlar: MyDrive > TourFold_TSP > v3. Bağlantı koparsa hücreyi
# yeniden çalıştırın; biten adımlar atlanır. İlerleme ve kalan süre: TourFold_TSP/v3/log_v3.txt


import os, re, sys, json, time, math, glob, shutil, traceback, importlib, datetime, warnings

TEST_MODE = os.environ.get("TOURFOLD_TEST") == "1"          # yalnızca yerel testler için
if TEST_MODE:
    ROOT, LOCAL = os.environ["TOURFOLD_ROOT"], os.environ["TOURFOLD_LOCAL"]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, LOCAL = "/content/drive/MyDrive/TourFold_TSP", "/content/tourfold_local"
for _d in ["", "/eval", "/train", "/runs", "/results", "/figures"]:
    os.makedirs(ROOT + _d, exist_ok=True)
R2 = f"{ROOT}/v2"                     # v2 outputs (TourFold-v2 checkpoint, v2 runs); read only in v3
RES2 = f"{R2}/results"
R3 = f"{ROOT}/v3"                     # v3 outputs
RES3 = f"{R3}/results"
for _d in ["", "/results", "/figures"]:
    os.makedirs(R3 + _d, exist_ok=True)
for _d in ["", "/work", "/tsplib"]:
    os.makedirs(LOCAL + _d, exist_ok=True)

# Yardımcı modüller bu hücrenin içinde taşınır ve diske yazılır (süreç havuzlarının içe aktarabilmesi için).
MODULES = {"tsplab.py": r'''"""
tsplab - utilities for "learned guidance + LKH-3" TSP experiments.

  * instance generation (uniform / clustered / mixed), integer scaling
  * TSPLIB I/O: problem, tour, parameter, LKH candidate and penalty (pi) files
  * LKH-3 download/build, runner, output parser (anytime trace)
  * Held-Karp 1-tree bound with LKH's 1-tree definition (unit checks)
  * candidate-set coverage metrics
  * dataset shards for variable-size instances
  * process-parallel map with error capture
"""
from __future__ import annotations

import glob
import gzip
import io
import json
import math
import os
import re
import shutil
import subprocess
import tarfile
import tempfile
import time
import traceback
import urllib.request

import numpy as np

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    def tqdm(x, **kwargs):
        return x

COORD_SCALE = 1_000_000          # synthetic instances: unit square -> integer grid
LKH_PRECISION = 100              # LKH default PRECISION
CAND_K = 5                       # LKH default MAX_CANDIDATES

# Known optimal tour lengths (TSPLIB, symmetric, 1k-20k nodes).
TSPLIB_OPT = {
    "pr1002": 259045, "si1032": 92650, "u1060": 224094, "vm1084": 239297,
    "pcb1173": 56892, "d1291": 50801, "rl1304": 252948, "rl1323": 270199,
    "nrw1379": 56638, "fl1400": 20127, "u1432": 152970, "fl1577": 22249,
    "d1655": 62128, "vm1748": 336556, "u1817": 57201, "rl1889": 316536,
    "d2103": 80450, "u2152": 64253, "u2319": 234256, "pr2392": 378032,
    "pcb3038": 137694, "fl3795": 28772, "fnl4461": 182566, "rl5915": 565530,
    "rl5934": 556045, "pla7397": 23260728, "rl11849": 923288,
    "usa13509": 19982859, "brd14051": 469385, "d15112": 1573084,
    "d18512": 645238, "dsj1000": 18660188,
}


# ----------------------------------------------------------------------------- env
def env_info():
    info = {"nproc": os.cpu_count()}
    try:
        with open("/proc/meminfo") as f:
            kb = int(re.search(r"MemTotal:\s+(\d+)", f.read()).group(1))
        info["ram_gb"] = round(kb / 1024 ** 2, 1)
    except Exception:
        info["ram_gb"] = None
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=20).stdout.strip()
        info["gpu"] = out or None
    except Exception:
        info["gpu"] = None
    info["numpy"] = np.__version__
    return info


# ---------------------------------------------------------------------- generation
def normalize_unit(xy):
    xy = np.asarray(xy, dtype=np.float64)
    mn = xy.min(0)
    span = float((xy.max(0) - mn).max())
    return (xy - mn) / (span if span > 0 else 1.0)


def gen_uniform(n, rng):
    return rng.random((n, 2))


def gen_clustered(n, rng, n_clusters=None, sigma=None):
    c = int(n_clusters if n_clusters is not None else rng.integers(3, 16))
    s = float(sigma if sigma is not None else math.exp(rng.uniform(math.log(0.02), math.log(0.08))))
    centers = rng.random((c, 2))
    lab = rng.integers(0, c, n)
    return normalize_unit(centers[lab] + rng.normal(0.0, s, (n, 2)))


def gen_instance(n, dist, rng):
    if dist == "uniform":
        return gen_uniform(n, rng)
    if dist == "clustered":
        return gen_clustered(n, rng)
    if dist == "mixed":
        return gen_uniform(n, rng) if rng.random() < 0.5 else gen_clustered(n, rng)
    raise ValueError(dist)


def to_int_coords(xy01, scale=COORD_SCALE):
    return np.rint(np.asarray(xy01, dtype=np.float64) * scale).astype(np.int64)


# ------------------------------------------------------------------------ distances
def edge_len(a, b, ew="EUC_2D"):
    d = np.sqrt(((np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)) ** 2).sum(-1))
    if ew == "EUC_2D":
        return np.floor(d + 0.5)
    if ew == "CEIL_2D":
        return np.ceil(d)
    if ew in ("FLOAT", "EXACT"):
        return d
    raise ValueError(f"unsupported EDGE_WEIGHT_TYPE {ew}")


def tour_length(xy, tour, ew="EUC_2D"):
    t = np.asarray(tour)
    xy = np.asarray(xy)
    return float(edge_len(xy[t], xy[np.roll(t, -1)], ew).sum())


def is_valid_tour(tour, n):
    if tour is None:
        return False
    t = np.asarray(tour)
    return t.shape == (n,) and np.array_equal(np.sort(t), np.arange(n))


def gap_pct(cost, ref):
    return 100.0 * (cost - ref) / ref


# --------------------------------------------------------------------- TSPLIB I/O
def _open_text(path):
    if str(path).endswith(".gz"):
        return io.TextIOWrapper(gzip.open(path), encoding="latin-1")
    return open(path, encoding="latin-1")


def _is_int_token(s):
    try:
        int(s)
        return True
    except ValueError:
        return False


def write_tsp(path, xy, name="instance", ew="EUC_2D", comment=None):
    xy = np.asarray(xy)
    head = [f"NAME : {name}", "TYPE : TSP"]
    if comment:
        head.append(f"COMMENT : {comment}")
    head += [f"DIMENSION : {len(xy)}", f"EDGE_WEIGHT_TYPE : {ew}", "NODE_COORD_SECTION"]
    if np.issubdtype(xy.dtype, np.integer):
        body = "\n".join(f"{i + 1} {x} {y}" for i, (x, y) in enumerate(xy.tolist()))
    else:
        body = "\n".join(f"{i + 1} {x!r} {y!r}" for i, (x, y) in enumerate(xy.tolist()))
    with open(path, "w") as f:
        f.write("\n".join(head) + "\n" + body + "\nEOF\n")


def read_tsplib(path):
    hdr, ids, coords = {}, [], []
    section = None
    with _open_text(path) as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.upper() == "EOF":
                break
            up = line.upper()
            if section == "NODE_COORD_SECTION":
                parts = line.split()
                if len(parts) >= 3 and _is_int_token(parts[0]):
                    ids.append(int(parts[0]))
                    coords.append((float(parts[1]), float(parts[2])))
                    continue
                section = None
            if "_SECTION" in up.split(":")[0]:
                section = up.split(":")[0].strip()
                continue
            if section is not None:
                continue
            if ":" in line:
                k, v = line.split(":", 1)
                hdr[k.strip().upper()] = v.strip()
    xy = np.array(coords, dtype=np.float64)
    if ids:
        xy = xy[np.argsort(np.array(ids))]
    return {
        "name": hdr.get("NAME", os.path.basename(str(path)).split(".")[0]),
        "type": hdr.get("TYPE"),
        "dimension": int(hdr.get("DIMENSION", len(xy))),
        "edge_weight_type": hdr.get("EDGE_WEIGHT_TYPE"),
        "comment": hdr.get("COMMENT"),
        "xy": xy,
    }


def write_par(path, params):
    with open(path, "w") as f:
        for k, v in params.items():
            if v is None:
                continue
            for x in (v if isinstance(v, (list, tuple)) else [v]):
                if isinstance(x, bool):
                    x = "YES" if x else "NO"
                f.write(f"{k} = {x}\n")


def read_tour(path):
    vals, in_sec = [], False
    with _open_text(path) as f:
        for line in f:
            s = line.strip()
            if not in_sec:
                if s.upper().startswith("TOUR_SECTION"):
                    in_sec = True
                continue
            for tok in s.split():
                if tok.upper() == "EOF":
                    break
                v = int(tok)
                if v == -1:
                    return np.array(vals, dtype=np.int64) - 1
                vals.append(v)
    return np.array(vals, dtype=np.int64) - 1


def write_tour(path, tour, name="tour", length=None):
    t = np.asarray(tour, dtype=np.int64) + 1
    head = [f"NAME : {name}"]
    if length is not None:
        head.append(f"COMMENT : Length = {int(length)}")
    head += ["TYPE : TOUR", f"DIMENSION : {len(t)}", "TOUR_SECTION"]
    with open(path, "w") as f:
        f.write("\n".join(head) + "\n" + "\n".join(map(str, t.tolist())) + "\n-1\nEOF\n")


def write_candidates(path, cand, alpha=None, dad=None):
    """LKH CANDIDATE_FILE. cand: (n,K) 0-based node ids, -1 padded, best first.
    alpha: same shape integer ranks/scores (default rank*100). dad: (n,) 0-based or -1."""
    cand = np.asarray(cand, dtype=np.int64)
    n = len(cand)
    lines = [str(n)]
    for i in range(n):
        row = cand[i]
        valid = row >= 0
        row = row[valid]
        if alpha is None:
            al = np.arange(len(row), dtype=np.int64) * 100
        else:
            al = np.asarray(alpha[i], dtype=np.int64)[valid]
        d = 0 if dad is None or dad[i] < 0 else int(dad[i]) + 1
        pairs = np.empty(2 * len(row), dtype=np.int64)
        pairs[0::2] = row + 1
        pairs[1::2] = al
        lines.append(f"{i + 1} {d} {len(row)} " + " ".join(map(str, pairs.tolist())))
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n-1\nEOF\n")


def read_candidates(path, k_pad=None):
    with open(path) as f:
        toks = f.read().split()
    n = int(toks[0])
    pos = 1
    lists, alists = [None] * n, [None] * n
    dad = np.full(n, -1, dtype=np.int64)
    while pos < len(toks) and _is_int_token(toks[pos]):
        tid = int(toks[pos])
        pos += 1
        if tid == -1:
            break
        d, cnt = int(toks[pos]), int(toks[pos + 1])
        pos += 2
        v = np.array(toks[pos:pos + 2 * cnt], dtype=np.int64).reshape(cnt, 2)
        pos += 2 * cnt
        order = np.argsort(v[:, 1], kind="stable")
        lists[tid - 1], alists[tid - 1] = v[order, 0] - 1, v[order, 1]
        dad[tid - 1] = d - 1
    K = max([len(x) for x in lists if x is not None] + [0])
    K = max(K, k_pad or 0)
    C = np.full((n, K), -1, dtype=np.int64)
    A = np.full((n, K), -1, dtype=np.int64)
    for i, x in enumerate(lists):
        if x is not None and len(x):
            C[i, :len(x)] = x
            A[i, :len(x)] = alists[i]
    return {"n": n, "cand": C, "alpha": A, "dad": dad}


def write_pi(path, pi):
    pi = np.rint(np.asarray(pi, dtype=np.float64)).astype(np.int64)
    with open(path, "w") as f:
        f.write(f"{len(pi)}\n" + "\n".join(f"{i + 1} {v}" for i, v in enumerate(pi.tolist())) + "\n-1\nEOF\n")


def read_pi(path):
    with open(path) as f:
        toks = f.read().split()
    n = int(toks[0])
    pi = np.zeros(n, dtype=np.int64)
    pos = 1
    for _ in range(n):
        i, v = int(toks[pos]), int(toks[pos + 1])
        pos += 2
        pi[i - 1] = v
    return pi


# -------------------------------------------------------------------------- LKH
LKH_BASES = ["http://webhotel4.ruc.dk/~keld/research/LKH-3/",
             "https://webhotel4.ruc.dk/~keld/research/LKH-3/"]
LKH_FALLBACK_VERSIONS = ["3.0.13", "3.0.12", "3.0.11", "3.0.10", "3.0.9", "3.0.8"]


def _urlget(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (tsplab)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()


def _extract(tgz, dest):
    with tarfile.open(tgz) as tf:
        try:
            tf.extractall(dest, filter="data")
        except TypeError:
            tf.extractall(dest)


def build_lkh(dest="/content/lkh", version=None, verbose=True):
    """Download the latest LKH-3 from the author's site and compile it. Returns the binary path."""
    os.makedirs(dest, exist_ok=True)
    found = sorted(glob.glob(os.path.join(dest, "LKH-3*", "LKH")))
    if found and version is None:
        return found[-1]
    versions = [version] if version else []
    if not versions:
        for base in LKH_BASES:
            try:
                html = _urlget(base, 30).decode("latin-1")
                vs = sorted({tuple(map(int, m)) for m in re.findall(r"LKH-3\.(\d+)\.(\d+)\.tgz", html)},
                            reverse=True)
                versions = [f"3.{a}.{b}" for a, b in vs]
                if versions:
                    break
            except Exception as e:  # noqa: BLE001
                if verbose:
                    print("index fetch failed:", base, e)
        versions += [v for v in LKH_FALLBACK_VERSIONS if v not in versions]
    last = None
    for v in versions:
        for base in LKH_BASES:
            try:
                data = _urlget(f"{base}LKH-{v}.tgz", 180)
            except Exception as e:  # noqa: BLE001
                last = e
                continue
            tgz = os.path.join(dest, f"LKH-{v}.tgz")
            with open(tgz, "wb") as f:
                f.write(data)
            _extract(tgz, dest)
            src = os.path.join(dest, f"LKH-{v}")
            r = subprocess.run(["make"], cwd=src, capture_output=True, text=True)
            binp = os.path.join(src, "LKH")
            if r.returncode != 0 or not os.path.exists(binp):
                raise RuntimeError(f"LKH build failed:\n{r.stdout[-3000:]}\n{r.stderr[-3000:]}")
            if verbose:
                print(f"LKH-{v} built at {binp}")
            return binp
    raise RuntimeError(f"Could not download LKH-3 ({last})")


_RX = {
    "lb": re.compile(r"Lower bound\s*=\s*(-?[\d.]+)"),
    "ascent": re.compile(r"Ascent time\s*=\s*([\d.]+)\s*sec"),
    "pre": re.compile(r"Preprocessing time\s*=\s*([\d.]+)\s*sec"),
    "imp": re.compile(r"^\*\s*(\d+):\s*Cost\s*=\s*(-?\d+)[^\n]*?Time\s*=\s*([\d.]+)\s*sec", re.M),
    "run": re.compile(r"^Run\s+(\d+):\s*Cost\s*=\s*(-?\d+)[^\n]*?Time\s*=\s*([\d.]+)\s*sec", re.M),
    "costmin": re.compile(r"Cost\.min\s*=\s*(-?\d+)"),
    "timetotal": re.compile(r"Time\.total\s*=\s*([\d.]+)\s*sec"),
    "cand": re.compile(r"Cand\.min\s*=\s*(\d+),\s*Cand\.avg\s*=\s*([\d.]+),\s*Cand\.max\s*=\s*(\d+)"),
    "trials": re.compile(r"Trials\.min\s*=\s*(\d+)"),
}


def parse_lkh_output(text):
    def f1(key, cast=float):
        m = _RX[key].search(text)
        return cast(m.group(1)) if m else None

    out = {
        "lower_bound": f1("lb"),
        "ascent_time": f1("ascent"),
        "pre_time": f1("pre"),
        "cost_min": f1("costmin", int),
        "time_total": f1("timetotal"),
        "trials": f1("trials", int),
        "events": [(int(a), int(b), float(c)) for a, b, c in _RX["imp"].findall(text)],
        "runs": [(int(a), int(b), float(c)) for a, b, c in _RX["run"].findall(text)],
    }
    m = _RX["cand"].search(text)
    out["cand_stats"] = (int(m.group(1)), float(m.group(2)), int(m.group(3))) if m else None
    return out


def run_lkh(lkh_bin, problem_file, params, workdir, timeout=None, keep_stdout=False):
    os.makedirs(workdir, exist_ok=True)
    par_path = os.path.join(workdir, "run.par")
    p = {"PROBLEM_FILE": problem_file, "RUNS": 1, "TRACE_LEVEL": 1}
    p.update(params or {})
    p.setdefault("OUTPUT_TOUR_FILE", os.path.join(workdir, "run.tour"))
    write_par(par_path, p)
    t0 = time.perf_counter()
    try:
        proc = subprocess.run([lkh_bin, par_path], cwd=workdir, stdin=subprocess.DEVNULL,
                              capture_output=True, text=True, errors="replace", timeout=timeout)
        rc, so, se, to = proc.returncode, proc.stdout, proc.stderr, False
    except subprocess.TimeoutExpired as e:
        so = e.stdout.decode("latin-1", "replace") if isinstance(e.stdout, bytes) else (e.stdout or "")
        se = e.stderr.decode("latin-1", "replace") if isinstance(e.stderr, bytes) else (e.stderr or "")
        rc, to = None, True
    wall = time.perf_counter() - t0
    res = parse_lkh_output(so)
    res.update(wall=wall, returncode=rc, timed_out=to, stderr=(se or "")[-2000:])
    res["stdout"] = so if keep_stdout else so[-4000:]
    tp = p["OUTPUT_TOUR_FILE"]
    res["tour"] = read_tour(tp) if os.path.exists(tp) else None
    return res


def lkh_solve(lkh_bin, xy=None, problem_file=None, ew="EUC_2D", params=None, workroot=None,
              want_cands=False, want_pi=False, cand_file=None, pi_file=None, timeout=None,
              keep_stdout=False):
    """Run LKH on coordinates (written to a temp TSPLIB file) or on an existing problem file.
    cand_file / pi_file: existing files to be READ by LKH (learned guidance).
    want_cands / want_pi: let LKH WRITE its own candidate set / penalties and return them."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="lkh_")
    try:
        if problem_file is None:
            problem_file = os.path.join(wd, "p.tsp")
            write_tsp(problem_file, xy, ew=ew)
        p = dict(params or {})
        out_cand = out_pi = None
        if cand_file is not None:
            p["CANDIDATE_FILE"] = cand_file
        elif want_cands:
            out_cand = p["CANDIDATE_FILE"] = os.path.join(wd, "out.cand")
        if pi_file is not None:
            p["PI_FILE"] = pi_file
        elif want_pi:
            out_pi = p["PI_FILE"] = os.path.join(wd, "out.pi")
        res = run_lkh(lkh_bin, problem_file, p, wd, timeout=timeout, keep_stdout=keep_stdout)
        if out_cand and os.path.exists(out_cand):
            res["cand"] = read_candidates(out_cand, k_pad=p.get("MAX_CANDIDATES", CAND_K))
        if out_pi and os.path.exists(out_pi):
            res["pi"] = read_pi(out_pi)
        return res
    finally:
        shutil.rmtree(wd, ignore_errors=True)


def anytime_curve(res, ref, budgets):
    """Gap (%) of the best tour found within each total-time budget (preprocessing + search)."""
    pre = res.get("pre_time") or 0.0
    ev = sorted(res.get("events") or [], key=lambda e: e[2])
    out = []
    for b in budgets:
        best = None
        for _, c, t in ev:
            if pre + t <= b:
                best = c if best is None else min(best, c)
        out.append(gap_pct(best, ref) if best is not None else np.nan)
    return out


TSPLIB_BASES = ["https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/",
                "http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/"]


def _valid_tsp(path):
    try:
        if os.path.getsize(path) == 0:
            return False
        inst = read_tsplib(path)
        return inst["dimension"] > 0 and (len(inst["xy"]) == inst["dimension"] or inst["edge_weight_type"] == "EXPLICIT")
    except Exception:  # noqa: BLE001
        return False


def _local_tsplib(dest, names):
    """{name: dest/name.tsp} for every name found under dest as a valid .tsp or .tsp.gz (any subfolder).
    Empty files (left behind by interrupted downloads) are removed so that they never count as present."""
    have = {}
    for nm in names:
        out = os.path.join(dest, nm + ".tsp")
        if os.path.exists(out):
            if _valid_tsp(out):
                have[nm] = out
                continue
            os.remove(out)
        for cand in sorted(glob.glob(os.path.join(dest, "**", nm + ".tsp*"), recursive=True)):
            if cand == out or not (cand.endswith(".tsp") or cand.endswith(".tsp.gz")):
                continue
            try:
                if os.path.getsize(cand) == 0:
                    os.remove(cand)
                    continue
                if cand.endswith(".gz"):
                    with gzip.open(cand, "rb") as fi, open(out + ".part", "wb") as fo:
                        shutil.copyfileobj(fi, fo)
                    os.replace(out + ".part", out)
                else:
                    shutil.copy(cand, out)
                if _valid_tsp(out):
                    have[nm] = out
                    break
                os.remove(out)
            except Exception:  # noqa: BLE001
                continue
    return have


def download_tsplib(dest, names=None, verbose=True, timeout=30, bases=None):
    """TSPLIB problems as {name: path.tsp}.
    1) files already under dest (.tsp, .tsp.gz or ALL_tsp.tar.gz, e.g. copied there by the user);
    2) the official TSPLIB95 server. Each address is first probed with one small file and a short timeout,
       so an unreachable server costs about a minute instead of one long timeout per file."""
    os.makedirs(dest, exist_ok=True)
    names = list(names or TSPLIB_OPT)
    tgz = os.path.join(dest, "ALL_tsp.tar.gz")
    if os.path.exists(tgz):
        if os.path.getsize(tgz) == 0:
            os.remove(tgz)                      # left behind by an interrupted download
        else:
            try:
                _extract(tgz, dest)
            except Exception as e:  # noqa: BLE001
                if verbose:
                    print("could not extract", tgz, e)
    have = _local_tsplib(dest, names)
    if len(have) == len(names):
        return have
    for base in (bases or TSPLIB_BASES):
        probe = next(nm for nm in names if nm not in have)
        try:
            data = _urlget(base + probe + ".tsp.gz", timeout)
        except Exception as e:  # noqa: BLE001
            if verbose:
                print("TSPLIB server not answering at", base, "-", e)
            continue
        with open(os.path.join(dest, probe + ".tsp.gz"), "wb") as f:
            f.write(data)
        try:
            data = _urlget(base + "ALL_tsp.tar.gz", max(timeout, 120))
            with open(tgz + ".part", "wb") as f:
                f.write(data)
            os.replace(tgz + ".part", tgz)
            _extract(tgz, dest)
        except Exception as e:  # noqa: BLE001
            if verbose:
                print("ALL_tsp.tar.gz not available, fetching single files:", e)
        have = _local_tsplib(dest, names)
        for nm in [m for m in names if m not in have]:
            try:
                data = _urlget(base + nm + ".tsp.gz", timeout)
                with open(os.path.join(dest, nm + ".tsp.gz"), "wb") as f:
                    f.write(data)
            except Exception as e:  # noqa: BLE001
                if verbose:
                    print("could not get", nm, e)
        break
    return _local_tsplib(dest, names)


def time_to_gap(res, ref, target_pct):
    """Earliest total time (preprocessing + search) at which the gap reaches target_pct; inf if never."""
    pre = res.get("pre_time") or 0.0
    for _, c, t in sorted(res.get("events") or [], key=lambda e: e[2]):
        if gap_pct(c, ref) <= target_pct + 1e-12:
            return pre + t
    return float("inf")


# ------------------------------------------------------------- Held-Karp 1-tree
def _row_cost(xy, u, ew, precision, pi):
    r = edge_len(xy[u], xy, ew) * precision
    if pi is not None:
        r = r + pi[u] + pi
    return r


def held_karp_bound(xy, pi=None, ew="EUC_2D", precision=1.0, chunk=256):
    """w(pi) with LKH's minimum 1-tree: MST + the longest second-nearest edge among MST leaves.
    Transformed cost C'_ij = precision*d_ij + pi_i + pi_j.  Returns bound in distance units."""
    xy = np.asarray(xy, dtype=np.float64)
    n = len(xy)
    pia = None if pi is None else np.asarray(pi, dtype=np.float64)
    in_tree = np.zeros(n, dtype=bool)
    best = np.full(n, np.inf)
    parent = np.full(n, -1, dtype=np.int64)
    best[0] = 0.0
    total = 0.0
    for _ in range(n):
        u = int(np.argmin(best))
        total += best[u]
        in_tree[u] = True
        best[u] = np.inf
        row = _row_cost(xy, u, ew, precision, pia)
        upd = (~in_tree) & (row < best)
        best[upd] = row[upd]
        parent[upd] = u
    deg = np.bincount(parent[parent >= 0], minlength=n) + (parent >= 0)
    nbr = parent.copy()
    root_child = np.where(parent == 0)[0]
    if deg[0] == 1:
        nbr[0] = root_child[0]
    leaves = np.where(deg == 1)[0]
    best_val, best_leaf, best_to = -np.inf, -1, -1
    for s in range(0, len(leaves), chunk):
        L = leaves[s:s + chunk]
        D = edge_len(xy[L][:, None, :], xy[None, :, :], ew) * precision
        if pia is not None:
            D = D + pia[L][:, None] + pia[None, :]
        D[np.arange(len(L)), L] = np.inf
        D[np.arange(len(L)), nbr[L]] = np.inf
        j = D.argmin(1)
        v = D[np.arange(len(L)), j]
        k = int(v.argmax())
        if v[k] > best_val:
            best_val, best_leaf, best_to = float(v[k]), int(L[k]), int(j[k])
    deg = deg.copy()
    deg[best_leaf] += 1
    deg[best_to] += 1
    w = total + best_val - (2.0 * pia.sum() if pia is not None else 0.0)
    return {"bound": w / precision, "degree": deg, "parent": parent, "special": (best_leaf, best_to)}


# ----------------------------------------------------------------- candidates
def knn(xy, k):
    from scipy.spatial import cKDTree
    xy = np.asarray(xy, dtype=np.float64)
    n = len(xy)
    k = min(k, n - 1)
    d, idx = cKDTree(xy).query(xy, k=k + 1)
    self_mask = idx == np.arange(n)[:, None]
    no_self = ~self_mask.any(1)
    self_mask[no_self, -1] = True
    keep = ~self_mask
    return idx[keep].reshape(n, k), d[keep].reshape(n, k)


def tour_succ_pred(tour):
    t = np.asarray(tour)
    succ = np.empty(len(t), dtype=np.int64)
    pred = np.empty(len(t), dtype=np.int64)
    succ[t] = np.roll(t, -1)
    pred[t] = np.roll(t, 1)
    return succ, pred


def tour_edge_labels(nbr_idx, tour):
    """(n,k) bool: is edge (i, nbr_idx[i,j]) in the tour (either direction)."""
    succ, pred = tour_succ_pred(tour)
    return (nbr_idx == succ[:, None]) | (nbr_idx == pred[:, None])


def candidate_coverage(cand, tour, ks=None):
    cand = np.asarray(cand)
    n, K = cand.shape
    succ, pred = tour_succ_pred(tour)
    out = {}
    for k in (ks or range(1, K + 1)):
        c = cand[:, :k]
        in_s = (c == succ[:, None]).any(1)
        in_p = (c == pred[:, None]).any(1)
        und = in_s | in_p[succ]
        out[int(k)] = {"directed": float((in_s.sum() + in_p.sum()) / (2 * n)),
                       "undirected": float(und.mean()),
                       "missing": int(n - und.sum())}
    return out


def pad_cols(a, K, fill=-1):
    a = np.asarray(a)
    if a.shape[1] >= K:
        return a[:, :K]
    return np.concatenate([a, np.full((a.shape[0], K - a.shape[1]), fill, dtype=a.dtype)], 1)


# --------------------------------------------------------------------- shards
def save_shard(path, records):
    ns = np.array([int(r["n"]) for r in records], dtype=np.int64)
    data = {"offsets": np.concatenate([[0], np.cumsum(ns)])}
    for k, v0 in records[0].items():
        if k == "n":
            continue
        if isinstance(v0, np.ndarray) and v0.ndim >= 1 and len(v0) == records[0]["n"]:
            data["node__" + k] = np.concatenate([np.asarray(r[k]) for r in records], 0)
        else:
            vals = [np.nan if r.get(k) is None else r.get(k) for r in records]
            data["scal__" + k] = np.array(vals)
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = path[:-4] + ".tmp.npz" if path.endswith(".npz") else path + ".tmp.npz"
    np.savez_compressed(tmp, **data)
    os.replace(tmp, path)


class Shard:
    def __init__(self, path):
        z = np.load(path, allow_pickle=False)
        self.off = z["offsets"]
        self.node = {k[6:]: z[k] for k in z.files if k.startswith("node__")}
        self.scal = {k[6:]: z[k] for k in z.files if k.startswith("scal__")}

    def __len__(self):
        return len(self.off) - 1

    def __getitem__(self, i):
        a, b = int(self.off[i]), int(self.off[i + 1])
        r = {k: v[a:b] for k, v in self.node.items()}
        r.update({k: (v[i].item() if getattr(v[i], "ndim", 1) == 0 else v[i]) for k, v in self.scal.items()})
        r["n"] = b - a
        return r


def load_shards(pattern):
    out = []
    for p in sorted(glob.glob(pattern)):
        s = Shard(p)
        out.extend(s[i] for i in range(len(s)))
    return out


# ---------------------------------------------------------------- parallel map
def _safe_call(args):
    fn, kw = args
    try:
        return fn(**kw)
    except Exception as e:  # noqa: BLE001
        return {"error": repr(e), "traceback": traceback.format_exc()}


def pmap(fn, kwargs_list, workers=None, desc=None, ctx="fork"):
    """Process-parallel map; exceptions are returned as {'error': ...} instead of raised."""
    import concurrent.futures as cf
    import multiprocessing as mp
    workers = max(1, workers or os.cpu_count() or 1)
    if workers == 1 or len(kwargs_list) <= 1:
        return [_safe_call((fn, kw)) for kw in tqdm(kwargs_list, desc=desc)]
    out = [None] * len(kwargs_list)
    with cf.ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context(ctx)) as ex:
        futs = {ex.submit(_safe_call, (fn, kw)): i for i, kw in enumerate(kwargs_list)}
        for f in tqdm(cf.as_completed(futs), total=len(futs), desc=desc):
            out[futs[f]] = f.result()
    return out


def errors_of(results):
    return [r for r in results if isinstance(r, dict) and "error" in r]


def call_task(fn, kw):
    """Run fn(**kw). Lets a single pool run different task functions; picklable under 'spawn'
    (functions defined inside a notebook cell are not)."""
    return fn(**kw)


# ------------------------------------------------------------ task functions
def task_record(lkh_bin, workroot, seed, xy=None, problem_file=None, ew="EUC_2D", params=None,
                dist=None, name=None, require_files=True, ensemble_M=0):
    """Solve one instance with LKH and keep: tour, alpha-candidates, penalties (pi), timings.
    ensemble_M > 0 also stores a TourFold ensemble: M single-trial tours on shared POPMUSIC candidates."""
    p = {"SEED": int(seed), "MAX_CANDIDATES": CAND_K}
    p.update(params or {})
    res = lkh_solve(lkh_bin, xy=None if problem_file else xy, problem_file=problem_file, ew=ew,
                    params=p, workroot=workroot, want_cands=True, want_pi=True)
    n = len(xy)
    if not is_valid_tour(res["tour"], n):
        raise RuntimeError(f"invalid tour (rc={res['returncode']}): {res['stdout'][-600:]}")
    cost = tour_length(xy, res["tour"], ew)
    if res.get("cost_min") is not None and abs(cost - res["cost_min"]) > 0.5:
        raise RuntimeError(f"cost mismatch {cost} vs {res['cost_min']}")
    xy_store = np.asarray(xy, dtype=np.int32) if np.issubdtype(np.asarray(xy).dtype, np.integer) \
        else np.asarray(xy, dtype=np.float64)
    rec = {"n": n, "xy": xy_store, "tour": res["tour"].astype(np.int32), "cost": cost,
           "lower_bound": res["lower_bound"], "ascent_time": res["ascent_time"],
           "pre_time": res["pre_time"], "wall": res["wall"], "seed": int(seed),
           "dist": dist or "", "name": name or "", "ew": ew}
    if require_files and ("cand" not in res or "pi" not in res):
        raise RuntimeError("LKH did not write CANDIDATE_FILE/PI_FILE: " + res["stdout"][-600:])
    if "cand" in res:
        rec["cand"] = pad_cols(res["cand"]["cand"], CAND_K).astype(np.int32)
        rec["cand_alpha"] = pad_cols(res["cand"]["alpha"], CAND_K).astype(np.int64)
    if "pi" in res:
        rec["pi"] = res["pi"].astype(np.int64)
    if ensemble_M:
        ens = task_ensemble(lkh_bin, workroot, ensemble_M, seed0=int(seed) * 1000 + 17, xy=xy,
                            problem_file=problem_file, ew=ew)
        rec["ens_succ"] = ens["succ"]
        rec["ens_cost"] = ens["costs"].astype(np.float64)
        rec["ens_wall"] = float(ens["walls"].sum())
        rec["pop_cand"] = ens["cand"]
    return rec


def task_train_record(lkh_bin, xy_int, seed, workroot, params=None, dist=None):
    return task_record(lkh_bin, workroot, seed, xy=xy_int, params=params, dist=dist)


def task_candidates(lkh_bin, workroot, xy=None, problem_file=None, ew="EUC_2D", ctype="ALPHA",
                    subgradient=False, k=CAND_K, seed=1):
    p = {"CANDIDATE_SET_TYPE": ctype, "SUBGRADIENT": "YES" if subgradient else "NO",
         "MAX_CANDIDATES": k, "MAX_TRIALS": 1, "SEED": seed}
    res = lkh_solve(lkh_bin, xy=xy, problem_file=problem_file, ew=ew, params=p, workroot=workroot,
                    want_cands=True)
    if "cand" not in res:
        raise RuntimeError(f"no candidate file written ({ctype}): {res['stdout'][-800:]}")
    return {"cand": pad_cols(res["cand"]["cand"], k), "pre_time": res["pre_time"],
            "wall": res["wall"], "cand_stats": res["cand_stats"]}


def task_anytime(lkh_bin, workroot, budget, seed, xy=None, problem_file=None, ew="EUC_2D",
                 params=None, cand_file=None, pi_file=None, timeout=None):
    """timeout: wall-clock cap in seconds (LKH is stopped there; the trace printed so far is kept). LKH's own
    TIME_LIMIT does not count preprocessing, so a run with a long ascent needs the cap to stay within budget."""
    p = {"TIME_LIMIT": float(budget), "MAX_TRIALS": 100_000_000, "SEED": int(seed)}
    p.update(params or {})
    res = lkh_solve(lkh_bin, xy=xy, problem_file=problem_file, ew=ew, params=p, workroot=workroot,
                    cand_file=cand_file, pi_file=pi_file, timeout=budget * 3 + 600 if timeout is None else timeout)
    return {k: res.get(k) for k in ("events", "pre_time", "ascent_time", "lower_bound", "wall",
                                    "cost_min", "time_total", "cand_stats", "returncode",
                                    "timed_out")}


# ------------------------------------------------------------ tour ensembles (TourFold)
def task_ensemble(lkh_bin, workroot, M, seed0, xy, problem_file=None, ew="EUC_2D", ctype="POPMUSIC",
                  k=CAND_K, extra_params=None):
    """M independent single-trial LKH tours that share one ascent-free candidate set.
    Run 0 builds the candidate set (POPMUSIC by default) and writes it; runs 1..M-1 read it together with
    zero penalties, so no run pays for a subgradient ascent. Returns successor arrays (n, M), tour costs,
    per-run wall times (sequential, one core), the shared candidate set and the best tour."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="ens_")
    try:
        n = len(xy)
        if problem_file is None:
            problem_file = os.path.join(wd, "p.tsp")
            write_tsp(problem_file, xy, ew=ew)
        cand_path, pi_path = os.path.join(wd, "c.cand"), os.path.join(wd, "zero.pi")
        base = {"MAX_TRIALS": 1, "SUBGRADIENT": "NO", "MAX_CANDIDATES": k}
        base.update(extra_params or {})
        tours, costs, walls = [], [], []
        cand = None
        for m in range(M):
            p = dict(base, SEED=int(seed0) + m)
            if m == 0:
                p.update(CANDIDATE_SET_TYPE=ctype, CANDIDATE_FILE=cand_path)
            else:
                p.update(CANDIDATE_FILE=cand_path, PI_FILE=pi_path)
            res = run_lkh(lkh_bin, problem_file, p, os.path.join(wd, f"r{m}"))
            if not is_valid_tour(res["tour"], n):
                raise RuntimeError(f"ensemble run {m} gave no valid tour: {res['stdout'][-500:]}")
            if m == 0:
                if not os.path.exists(cand_path):
                    raise RuntimeError("LKH did not write the ensemble candidate file: " + res["stdout"][-500:])
                cand = read_candidates(cand_path, k_pad=k)
                write_pi(pi_path, np.zeros(n))
            tours.append(res["tour"])
            costs.append(tour_length(xy, res["tour"], ew))
            walls.append(res["wall"])
        succ = np.stack([tour_succ_pred(t)[0] for t in tours], 1).astype(np.int32)
        b = int(np.argmin(costs))
        return {"succ": succ, "costs": np.array(costs), "walls": np.array(walls),
                "cand": pad_cols(cand["cand"], k).astype(np.int32), "best": tours[b].astype(np.int32),
                "best_cost": float(costs[b])}
    finally:
        shutil.rmtree(wd, ignore_errors=True)


def ensemble_pred(succ):
    n, M = succ.shape
    pred = np.empty_like(succ)
    ar = np.arange(n, dtype=succ.dtype)
    for m in range(M):
        pred[succ[:, m], m] = ar
    return pred


def ensemble_frequency(nb, succ):
    """freq[i, a] = share of ensemble tours that use edge (i, nb[i, a]) in either direction."""
    pred = ensemble_pred(succ)
    hit = (nb[:, :, None] == succ[:, None, :]) | (nb[:, :, None] == pred[:, None, :])
    return hit.mean(-1)


def frequency_candidates(nb, d, freq, k=CAND_K):
    """'Count' baseline: rank each node's kNN edges by ensemble frequency, ties broken by distance."""
    order = np.lexsort((d, -freq), axis=1)
    return np.take_along_axis(nb, order[:, :k], 1)


def mix_candidates(first, rest, k_first=3, k=CAND_K, fill=None):
    """Mixed candidate lists. Per node: the first k_first entries of `first`, then the entries of `rest` (in
    their order) that are not in the list yet, then - if `fill` is given - entries of `fill` not in the list yet,
    until k entries. Inputs are (n, *) arrays of 0-based node ids, -1 padded; a node never lists itself.
    Returns (n, k), -1 padded where fewer than k distinct neighbours are available."""
    first = np.asarray(first, dtype=np.int64)
    n = len(first)
    parts = [first[:, :k_first], np.asarray(rest, dtype=np.int64).reshape(n, -1)]
    if fill is not None:
        parts.append(np.asarray(fill, dtype=np.int64).reshape(n, -1))
    L = np.concatenate(parts, 1)
    if L.shape[1] < k:
        L = np.concatenate([L, np.full((n, k - L.shape[1]), -1, dtype=np.int64)], 1)
    W = L.shape[1]
    dup = ((L[:, :, None] == L[:, None, :]) & np.tri(W, W, -1, dtype=bool)[None]).any(2)   # seen earlier in row
    valid = (L >= 0) & ~dup & (L != np.arange(n)[:, None])
    order = np.argsort(~valid, axis=1, kind="stable")[:, :k]
    return np.where(np.take_along_axis(valid, order, 1), np.take_along_axis(L, order, 1), -1)


def fixing_stats(nb, score, tour, thresholds):
    """Precision / recall of 'fix edge if score >= t' against a reference tour, per threshold.
    Edges are undirected: an edge counts once and is fixed if either direction passes."""
    n, K = nb.shape
    lab = tour_edge_labels(nb, tour)
    i = np.repeat(np.arange(n), K)
    j = nb.reshape(-1)
    key = np.minimum(i, j) * n + np.maximum(i, j)
    s, y = score.reshape(-1), lab.reshape(-1)
    uniq, inv = np.unique(key, return_inverse=True)
    best = np.full(len(uniq), -np.inf)
    np.maximum.at(best, inv, s)
    is_tour = np.zeros(len(uniq), dtype=bool)
    is_tour[inv[y]] = True
    out = []
    for t in thresholds:
        sel = best >= t
        tp, nf = int((sel & is_tour).sum()), int(sel.sum())
        out.append({"t": float(t), "fixed": nf, "tp": tp, "n": int(n),
                    "precision": tp / nf if nf else float("nan"), "recall": tp / n})
    return out


# ------------------------------------------------------------ guided LKH runs (TourFold evaluation)
def select_fixed_edges(nb, prob, tau):
    """Greedy, consistent set of edges to fix: highest probability first, degree <= 2, no cycles.
    prob[i, a] is the probability of edge (i, nb[i, a]); both directions are merged by max."""
    n, K = nb.shape
    i = np.repeat(np.arange(n), K)
    j = nb.reshape(-1)
    p = prob.reshape(-1)
    lo, hi = np.minimum(i, j), np.maximum(i, j)
    key = lo * n + hi
    uniq, inv = np.unique(key, return_inverse=True)
    best = np.full(len(uniq), -np.inf)
    np.maximum.at(best, inv, p)
    order = np.argsort(-best, kind="stable")
    parent = np.arange(n)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    deg = np.zeros(n, dtype=np.int64)
    out = []
    for e in order:
        if best[e] < tau or len(out) >= n - 1:
            break
        a, c = int(uniq[e] // n), int(uniq[e] % n)
        if deg[a] >= 2 or deg[c] >= 2:
            continue
        ra, rc = find(a), find(c)
        if ra == rc:
            continue
        parent[ra] = rc
        deg[a] += 1
        deg[c] += 1
        out.append((a, c))
    return np.array(out, dtype=np.int64).reshape(-1, 2)


def add_fixed_edges(src_problem, dst_problem, edges):
    """Copy a TSPLIB problem file and append a FIXED_EDGES_SECTION (0-based edges in, 1-based out)."""
    with _open_text(src_problem) as f:
        lines = [ln.rstrip("\n") for ln in f]
    while lines and lines[-1].strip() in ("", "EOF"):
        lines.pop()
    body = "\n".join(f"{a + 1} {b + 1}" for a, b in np.asarray(edges).tolist())
    with open(dst_problem, "w") as f:
        f.write("\n".join(lines) + "\nFIXED_EDGES_SECTION\n" + (body + "\n" if body else "") + "-1\nEOF\n")


def task_guided(lkh_bin, workroot, budget, seed, xy, cand, pi=None, init_tour=None, problem_file=None,
                ew="EUC_2D", fixed_edges=None, alpha=None, extra_params=None, return_tour=False, timeout=None):
    """LKH with externally supplied candidates (+ penalties, initial tour, fixed edges); no ascent."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="g_")
    try:
        n = len(xy)
        base = problem_file
        if base is None:
            base = os.path.join(wd, "p.tsp")
            write_tsp(base, xy, ew=ew)
        pf = base
        if fixed_edges is not None and len(fixed_edges):
            pf = os.path.join(wd, "fixed.tsp")
            add_fixed_edges(base, pf, fixed_edges)
        cf, pif = os.path.join(wd, "g.cand"), os.path.join(wd, "g.pi")
        write_candidates(cf, cand, alpha)
        write_pi(pif, np.zeros(n) if pi is None else pi)
        params = {"SUBGRADIENT": "NO", "TIME_LIMIT": float(budget), "MAX_TRIALS": 100_000_000,
                  "SEED": int(seed), "MAX_CANDIDATES": int(np.asarray(cand).shape[1])}
        if init_tour is not None:
            tp = os.path.join(wd, "init.tour")
            write_tour(tp, init_tour)
            params["INITIAL_TOUR_FILE"] = tp
        params.update(extra_params or {})
        params["CANDIDATE_FILE"], params["PI_FILE"] = cf, pif
        res = run_lkh(lkh_bin, pf, params, os.path.join(wd, "run"),
                      timeout=budget * 3 + 600 if timeout is None else timeout)
        ok = is_valid_tour(res["tour"], n)
        cost = tour_length(xy, res["tour"], ew) if ok else None
        out = {k: res.get(k) for k in ("events", "pre_time", "wall", "cost_min", "returncode", "timed_out")}
        out["cost_check"] = cost
        if fixed_edges is not None and len(fixed_edges):
            # LKH gives fixed edges length 0, so its reported costs miss their total length. Every tour of the
            # run contains all fixed edges, hence one constant shift restores true lengths for all events.
            fe = np.asarray(fixed_edges, dtype=np.int64).reshape(-1, 2)
            xa = np.asarray(xy)
            out["fixed_len"] = float(edge_len(xa[fe[:, 0]], xa[fe[:, 1]], ew).sum())
            if cost is not None and res.get("cost_min") is not None:
                shift = float(cost - res["cost_min"])
            else:
                shift = out["fixed_len"]
            out["cost_shift"] = shift
            out["events"] = [(i, int(round(c + shift)), t) for i, c, t in (res.get("events") or [])]
            if res.get("cost_min") is not None:
                out["cost_min"] = int(round(res["cost_min"] + shift))
        if not ok:
            out["stdout_tail"] = (res.get("stdout") or "")[-800:]
        if return_tour:
            out["tour"] = res["tour"] if ok else None
        return out
    finally:
        shutil.rmtree(wd, ignore_errors=True)


def abs_events(res, offset=0.0):
    """(cost, absolute time) pairs: offset + LKH preprocessing + time inside the run."""
    pre = (res.get("pre_time") or 0.0) + offset
    return [(int(c), pre + float(t)) for _, c, t in (res.get("events") or [])]


def curve_abs(events, ref, budgets):
    ev = sorted(events, key=lambda e: e[1])
    out = []
    for b in budgets:
        cs = [c for c, t in ev if t <= b]
        out.append(gap_pct(min(cs), ref) if cs else np.nan)
    return out


def time_to_gap_abs(events, ref, target_pct):
    for c, t in sorted(events, key=lambda e: e[1]):
        if gap_pct(c, ref) <= target_pct + 1e-12:
            return t
    return float("inf")
''', "tsp_model.py": r'''"""
tsp_model - AlphaFold-inspired sparse pair model ("PairFormer-lite") that predicts
LKH candidate edges and node penalties (pi) for Euclidean TSP.

Representation (as in AlphaFold's Evoformer):
  * single (node) representation  h_i            (N, dn)
  * pair (edge) representation    z_ij on kNN    (N, K, de), directed edges i -> NB[i, a]
Block = pair-biased node attention -> node transition -> node->edge product update ->
        reverse-edge mixing -> triangle multiplicative update (outgoing, incoming) ->
        [optional triangle attention] -> edge transition.
Triangles are the directed kNN triangles {i->j, i->k, j->k}; the same list serves the
"outgoing" (z_ij += sum_k a_ik * b_jk) and "incoming" (z_jk += sum_i a_ij * b_ik) updates.
All input features are invariant to translation, rotation, scale and node order.
Heads: edge logits (top-k -> CANDIDATE_FILE) and scale-equivariant node penalties
pi_i = s_i * f_i (-> PI_FILE). Optional AlphaFold-style recycling.
"""
from __future__ import annotations

import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

PRECISION = 100  # LKH default


# ------------------------------------------------------------------ graph + features
def knn_np(xy, K):
    from scipy.spatial import cKDTree
    n = len(xy)
    d, idx = cKDTree(xy).query(xy, k=K + 1)
    self_mask = idx == np.arange(n)[:, None]
    no_self = ~self_mask.any(1)
    self_mask[no_self, -1] = True
    keep = ~self_mask
    return idx[keep].reshape(n, K), d[keep].reshape(n, K)


def extend_graph(xyn, NB, ens_succ, extra):
    """TourFold v2 graph: kNN-K plus up to `extra` neighbours that the tour ensemble uses but kNN-K lacks
    (long edges, e.g. between clusters). They replace the farthest kNN slots, so every row still has K entries;
    the most frequently used ensemble edges come first. Rows are re-sorted by distance.
    Returns NB (n,K), D (n,K) and a flag (n,K) marking the ensemble-only edges."""
    n, K = NB.shape
    succ = np.asarray(ens_succ, dtype=np.int64)
    Mt = succ.shape[1]
    pred = np.empty_like(succ)
    ar = np.arange(n)
    for m in range(Mt):
        pred[succ[:, m], m] = ar
    cand = np.concatenate([succ, pred], 1)                              # (n, 2M) ensemble neighbours
    cand = np.where((cand[:, :, None] == NB[:, None, :]).any(-1), -1, cand)
    S = np.sort(cand, 1)
    first = np.ones_like(S, dtype=bool)
    first[:, 1:] = S[:, 1:] != S[:, :-1]
    valid = (S >= 0) & first
    cnt = (S[:, :, None] == S[:, None, :]).sum(-1)
    dist = np.linalg.norm(xyn[np.maximum(S, 0)] - xyn[:, None, :], axis=-1)
    score = np.where(valid, cnt * 10.0 - dist, -np.inf)                 # frequency first, then distance
    order = np.argsort(-score, 1, kind="stable")[:, :extra]
    chosen = np.take_along_axis(S, order, 1)
    nv = np.isfinite(np.take_along_axis(score, order, 1)).sum(1)       # ensemble-only neighbours used
    pos = np.arange(K)[None, :]
    keep = pos < (K - nv)[:, None]
    tail = np.take_along_axis(chosen, np.clip(pos - (K - nv)[:, None], 0, extra - 1), 1)
    NB2 = np.where(keep, NB, tail)
    D2 = np.linalg.norm(xyn[NB2] - xyn[:, None, :], axis=-1)
    o = np.argsort(D2, 1, kind="stable")
    return (np.take_along_axis(NB2, o, 1), np.take_along_axis(D2, o, 1),
            np.take_along_axis(~keep, o, 1))


def prepare_instance(xy, K, tour=None, pi=None, precision=PRECISION, ens_succ=None, ens_cost=None, extra=0):
    """Normalize coordinates, build kNN graph, labels and pi targets (numpy).
    extra > 0 (TourFold v2): the graph also gets up to `extra` ensemble edges per node (see extend_graph)."""
    xy = np.asarray(xy, dtype=np.float64)
    n = len(xy)
    if n <= K:
        raise ValueError(f"n={n} must exceed K={K}")
    mn = xy.min(0)
    span = float((xy.max(0) - mn).max()) or 1.0
    xyn = (xy - mn) / span
    NB, D = knn_np(xyn, K)
    s = np.maximum(D[:, : max(1, K // 2)].mean(1), 1e-9)
    ens_only = None
    if extra and ens_succ is not None:
        NB, D, ens_only = extend_graph(xyn, NB, ens_succ, int(extra))
    out = {"n": n, "span": span, "xy": xyn.astype(np.float32), "NB": NB.astype(np.int64),
           "D": D.astype(np.float32), "s": s.astype(np.float32),
           "s_glob": np.full(n, np.median(s), dtype=np.float32)}
    if ens_only is not None:
        out["ens_only"] = ens_only.astype(np.float32)
    if tour is not None:
        t = np.asarray(tour)
        succ = np.empty(n, np.int64)
        pred = np.empty(n, np.int64)
        succ[t] = np.roll(t, -1)
        pred[t] = np.roll(t, 1)
        out["y"] = (NB == succ[:, None]) | (NB == pred[:, None])
    if pi is not None:
        out["pi_t"] = (np.asarray(pi, dtype=np.float64) / (precision * span * s)).astype(np.float32)
    if ens_succ is not None:
        out.update(ensemble_features(xyn, NB, s, ens_succ, ens_cost))
    return out


def ensemble_features(xyn, NB, s, ens_succ, ens_cost=None):
    """TourFold 'tour alignment' (the MSA analogue): per tour m and node i, how tour m connects i.
    Everything is symmetric in the two tour neighbours, so a reversed tour gives the same features."""
    succ = np.asarray(ens_succ, dtype=np.int64)
    n, M = succ.shape
    K = NB.shape[1]
    pred = np.empty_like(succ)
    ar = np.arange(n)
    for m in range(M):
        pred[succ[:, m], m] = ar
    eq_s = NB[:, :, None] == succ[:, None, :]
    eq_p = NB[:, :, None] == pred[:, None, :]
    pos_s = np.where(eq_s.any(1), eq_s.argmax(1), K)
    pos_p = np.where(eq_p.any(1), eq_p.argmax(1), K)
    ds = np.log(np.linalg.norm(xyn[succ] - xyn[:, None, :], axis=-1) / s[:, None] + 1e-4)
    dp = np.log(np.linalg.norm(xyn[pred] - xyn[:, None, :], axis=-1) / s[:, None] + 1e-4)
    fs = ((succ[:, None, :] == succ[:, :, None]) | (pred[:, None, :] == succ[:, :, None])).mean(-1)
    fp = ((succ[:, None, :] == pred[:, :, None]) | (pred[:, None, :] == pred[:, :, None])).mean(-1)
    cost = np.ones(M) if ens_cost is None else np.asarray(ens_cost, dtype=np.float64)
    q = (cost - cost.min()) / max(cost.mean(), 1e-9) * 100.0
    msa_f = np.stack([np.minimum(ds, dp), np.maximum(ds, dp), np.broadcast_to(q[None, :], (n, M)),
                      np.minimum(fs, fp), np.maximum(fs, fp)], -1)
    hit = eq_s | eq_p
    top = np.argsort(cost, kind="stable")[: max(1, M // 2)]
    ef_ens = np.stack([hit.mean(-1), hit[:, :, top].mean(-1)], -1)
    return {"msa_f": msa_f.transpose(1, 0, 2).astype(np.float32),
            "msa_ps": pos_s.T.astype(np.int64), "msa_pp": pos_p.T.astype(np.int64),
            "ens_succ": succ.T.copy(), "ens_pred": pred.T.copy(),
            "ef_ens": ef_ens.astype(np.float32)}


def collate(items):
    """Concatenate prepared instances into one disjoint graph."""
    off, N = [], 0
    for it in items:
        off.append(N)
        N += it["n"]
    cat = lambda k: np.concatenate([it[k] for it in items], 0)  # noqa: E731
    b = {"xy": torch.from_numpy(cat("xy")), "D": torch.from_numpy(cat("D")),
         "s": torch.from_numpy(cat("s")), "s_glob": torch.from_numpy(cat("s_glob")),
         "NB": torch.from_numpy(np.concatenate([it["NB"] + o for it, o in zip(items, off)], 0)),
         "bidx": torch.from_numpy(np.concatenate([np.full(it["n"], i) for i, it in enumerate(items)])),
         "offsets": torch.tensor(off + [N]), "span": torch.tensor([it["span"] for it in items]),
         "B": len(items)}
    if all("y" in it for it in items):
        b["y"] = torch.from_numpy(cat("y"))
    if all("pi_t" in it for it in items):
        b["pi_t"] = torch.from_numpy(cat("pi_t"))
    if all("ens_only" in it for it in items):
        b["ens_only"] = torch.from_numpy(cat("ens_only"))
    if all("msa_f" in it for it in items):
        Ms = {it["msa_f"].shape[0] for it in items}
        if len(Ms) != 1:
            raise ValueError(f"all instances in a batch need the same ensemble size, got {Ms}")
        b["msa_f"] = torch.from_numpy(np.concatenate([it["msa_f"] for it in items], 1))
        b["msa_ps"] = torch.from_numpy(np.concatenate([it["msa_ps"] for it in items], 1))
        b["msa_pp"] = torch.from_numpy(np.concatenate([it["msa_pp"] for it in items], 1))
        b["ens_succ"] = torch.from_numpy(np.concatenate([it["ens_succ"] + o for it, o in zip(items, off)], 1))
        b["ens_pred"] = torch.from_numpy(np.concatenate([it["ens_pred"] + o for it, o in zip(items, off)], 1))
        b["ef_ens"] = torch.from_numpy(cat("ef_ens"))
    return b


def to_device(b, device):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}


@torch.no_grad()
def graph_context(b, k_tri=None, chunk_nodes=40000):
    """Invariant features, reverse-edge index and directed triangles (all on device).
    k_tri: the third node k of a triangle must be among the first k_tri neighbours of both i and j
    (fewer, more local triangles -> less memory)."""
    xy, NB, D, s, sg = b["xy"], b["NB"], b["D"], b["s"], b["s_glob"]
    N, K = NB.shape
    kt = K if not k_tri else min(int(k_tri), K)
    ar = torch.arange(N, device=NB.device)
    eq = NB[NB] == ar[:, None, None]                     # (N,K,K): NB[NB[i,a], c] == i
    mutual = eq.any(-1)
    rev_pos = eq.float().argmax(-1)
    rank = torch.arange(K, device=NB.device, dtype=xy.dtype).expand(N, K) / K
    rev_rank = torch.where(mutual, rev_pos.to(xy.dtype) / K, torch.full_like(rank, 1.5))
    vec = xy[NB] - xy[:, None, :]
    ang = torch.atan2(vec[..., 1], vec[..., 0])
    rel = ang - ang[:, :1]
    d1 = D / s[:, None]
    ef = torch.stack([d1, D / s[NB], D / sg[:, None], torch.log(d1 + 1e-4), rank, mutual.to(xy.dtype),
                      rev_rank, torch.cos(rel), torch.sin(rel).abs()], -1)
    if "ef_ens" in b:
        ef = torch.cat([ef, b["ef_ens"]], -1)
    if "ens_only" in b:
        ef = torch.cat([ef, b["ens_only"][..., None].to(ef.dtype)], -1)
    cen = xy[NB].mean(1)
    indeg = torch.bincount(NB.reshape(-1), minlength=N).to(xy.dtype) / K
    nf = torch.stack([s / sg, D[:, 0] / s, D[:, -1] / s, (cen - xy).norm(dim=-1) / s, indeg], -1)
    tris = []
    step = max(1, chunk_nodes // max(1, K * kt * kt // 4096))
    for st in range(0, N, step):
        nb = NB[st:st + step]
        M = NB[nb][:, :, None, :kt] == nb[:, None, :kt, None]   # (n,a,b,c): NB[NB[i,a],c] == NB[i,b]
        idx = M.nonzero()
        i = idx[:, 0] + st
        a, bb, c = idx[:, 1], idx[:, 2], idx[:, 3]
        tris.append(torch.stack([i * K + a, i * K + bb, NB[i, a] * K + c]))
    tri = torch.cat(tris, 1) if tris else torch.zeros(3, 0, dtype=torch.long, device=NB.device)
    return {"node_feat": nf, "edge_feat": ef, "mutual": mutual, "rev_pos": rev_pos, "tri": tri}


# ---------------------------------------------------------------------------- modules
class Transition(nn.Module):
    def __init__(self, d, mult=2):
        super().__init__()
        self.ln = nn.LayerNorm(d)
        self.l1 = nn.Linear(d, d * mult)
        self.l2 = nn.Linear(d * mult, d)

    def forward(self, x):
        return self.l2(F.gelu(self.l1(self.ln(x))))


class NodeAttention(nn.Module):
    """Attention of node i over its K neighbours, biased by the pair representation."""

    def __init__(self, dn, de, heads=4):
        super().__init__()
        self.h, self.dh = heads, dn // heads
        self.ln, self.lnz = nn.LayerNorm(dn), nn.LayerNorm(de)
        self.q, self.kv = nn.Linear(dn, dn), nn.Linear(dn, 2 * dn)
        self.bias = nn.Linear(de, heads)
        self.g, self.o = nn.Linear(dn, dn), nn.Linear(dn, dn)

    def forward(self, h, z, NB):
        N, K = NB.shape
        x = self.ln(h)
        q = self.q(x).view(N, self.h, self.dh)
        k, v = self.kv(x)[NB].split(self.h * self.dh, -1)
        k, v = k.view(N, K, self.h, self.dh), v.view(N, K, self.h, self.dh)
        logit = torch.einsum("nhd,nkhd->nhk", q, k) / math.sqrt(self.dh)
        logit = logit + self.bias(self.lnz(z)).permute(0, 2, 1)
        att = logit.float().softmax(-1).to(v.dtype)
        out = torch.einsum("nhk,nkhd->nhd", att, v).reshape(N, -1)
        return torch.sigmoid(self.g(x)) * self.o(out)


class EdgeFromNodes(nn.Module):
    def __init__(self, dn, de):
        super().__init__()
        self.ln = nn.LayerNorm(dn)
        self.l, self.r, self.o = nn.Linear(dn, de), nn.Linear(dn, de), nn.Linear(de, de)

    def forward(self, h, NB):
        x = self.ln(h)
        return self.o(self.l(x)[:, None, :] * self.r(x)[NB])


class ReverseMix(nn.Module):
    def __init__(self, de):
        super().__init__()
        self.ln, self.l = nn.LayerNorm(de), nn.Linear(de, de)

    def forward(self, z, NB, rev_pos, mutual):
        zr = z[NB, rev_pos]                                    # z_ji for edge i->j
        return self.l(self.ln(zr)) * mutual[..., None].to(z.dtype)


class TriangleMultiplication(nn.Module):
    """AlphaFold Alg. 11/12 on sparse directed kNN triangles."""

    def __init__(self, de, dh, mode):
        super().__init__()
        self.mode = mode
        self.ln = nn.LayerNorm(de)
        self.a, self.ag = nn.Linear(de, dh), nn.Linear(de, dh)
        self.b, self.bg = nn.Linear(de, dh), nn.Linear(de, dh)
        self.g = nn.Linear(de, de)
        self.ln_o, self.o = nn.LayerNorm(dh), nn.Linear(dh, de)

    def forward(self, z, tri):
        N, K, de = z.shape
        x = self.ln(z).reshape(N * K, de)
        a = torch.sigmoid(self.ag(x)) * self.a(x)
        b = torch.sigmoid(self.bg(x)) * self.b(x)
        e_ij, e_ik, e_jk = tri
        tgt, s1, s2 = (e_ij, e_ik, e_jk) if self.mode == "out" else (e_jk, e_ij, e_ik)
        prod = (a[s1] * b[s2]).float()
        m = torch.zeros(N * K, prod.shape[1], device=z.device, dtype=torch.float32).index_add_(0, tgt, prod)
        cnt = torch.bincount(tgt, minlength=N * K).clamp_min(1).float().sqrt()
        m = (m / cnt[:, None]).to(x.dtype)
        out = torch.sigmoid(self.g(x)) * self.o(self.ln_o(m))
        return out.view(N, K, de)


class TriangleAttention(nn.Module):
    """AlphaFold Alg. 13 (starting node) on sparse triangles: edge i->j attends over k with
    keys/values from i->k and a bias from j->k."""

    def __init__(self, de, heads=4):
        super().__init__()
        self.h = heads
        self.dh = max(8, de // heads)
        self.ln = nn.LayerNorm(de)
        self.q, self.k, self.v = (nn.Linear(de, self.h * self.dh) for _ in range(3))
        self.bias, self.g = nn.Linear(de, heads), nn.Linear(de, self.h * self.dh)
        self.o = nn.Linear(self.h * self.dh, de)

    def forward(self, z, tri):
        N, K, de = z.shape
        E = N * K
        x = self.ln(z).reshape(E, de)
        e_ij, e_ik, e_jk = tri
        q = self.q(x)[e_ij].view(-1, self.h, self.dh)
        k = self.k(x)[e_ik].view(-1, self.h, self.dh)
        logit = ((q * k).sum(-1) / math.sqrt(self.dh) + self.bias(x)[e_jk]).float()   # (T,h)
        mx = torch.full((E, self.h), -1e30, device=z.device).scatter_reduce(
            0, e_ij[:, None].expand(-1, self.h), logit, reduce="amax", include_self=True)
        w = torch.exp(logit - mx[e_ij])
        den = torch.zeros(E, self.h, device=z.device).index_add_(0, e_ij, w).clamp_min(1e-9)
        w = w / den[e_ij]
        v = self.v(x)[e_ik].view(-1, self.h, self.dh).float()
        out = torch.zeros(E, self.h, self.dh, device=z.device).index_add_(0, e_ij, w[..., None] * v)
        out = (torch.sigmoid(self.g(x)) * out.reshape(E, -1).to(x.dtype))
        return self.o(out).view(N, K, de)


class Block(nn.Module):
    def __init__(self, dn, de, dtri, heads, tri_attn):
        super().__init__()
        self.node_att = NodeAttention(dn, de, heads)
        self.node_tr = Transition(dn)
        self.e_from_n = EdgeFromNodes(dn, de)
        self.rev = ReverseMix(de)
        self.tri_out = TriangleMultiplication(de, dtri, "out")
        self.tri_in = TriangleMultiplication(de, dtri, "in")
        self.tri_att = TriangleAttention(de, heads) if tri_attn else None
        self.edge_tr = Transition(de)

    def forward(self, h, z, NB, rev_pos, mutual, tri):
        h = h + self.node_att(h, z, NB)
        h = h + self.node_tr(h)
        z = z + self.e_from_n(h, NB)
        z = z + self.rev(z, NB, rev_pos, mutual)
        z = z + self.tri_out(z, tri)
        z = z + self.tri_in(z, tri)
        if self.tri_att is not None:
            z = z + self.tri_att(z, tri)
        z = z + self.edge_tr(z)
        return h, z


class PairFormerTSP(nn.Module):
    def __init__(self, dn=128, de=64, dtri=32, layers=8, heads=4, tri_attn=False, recycles=0,
                 grad_ckpt=False, k_tri=12, node_in=5, edge_in=9):
        super().__init__()
        self.cfg = dict(dn=dn, de=de, dtri=dtri, layers=layers, heads=heads, tri_attn=tri_attn,
                        recycles=recycles, grad_ckpt=grad_ckpt, k_tri=k_tri, node_in=node_in, edge_in=edge_in)
        self.node_in = nn.Sequential(nn.Linear(node_in, dn), nn.GELU(), nn.Linear(dn, dn))
        self.edge_in = nn.Sequential(nn.Linear(edge_in, de), nn.GELU(), nn.Linear(de, de))
        self.rec_h = nn.Sequential(nn.LayerNorm(dn), nn.Linear(dn, dn))
        self.rec_z = nn.Sequential(nn.LayerNorm(de), nn.Linear(de, de))
        for m in (self.rec_h[1], self.rec_z[1]):
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)
        self.blocks = nn.ModuleList(Block(dn, de, dtri, heads, tri_attn) for _ in range(layers))
        self.edge_head = nn.Sequential(nn.LayerNorm(de), nn.Linear(de, de), nn.GELU(), nn.Linear(de, 1))
        self.pi_head = nn.Sequential(nn.LayerNorm(dn), nn.Linear(dn, dn), nn.GELU(), nn.Linear(dn, 1))
        self.grad_ckpt = grad_ckpt
        self.recycles = recycles

    def trunk(self, ctx, NB, h_prev=None, z_prev=None):
        h = self.node_in(ctx["node_feat"])
        z = self.edge_in(ctx["edge_feat"])
        if h_prev is not None:
            h = h + self.rec_h(h_prev)
            z = z + self.rec_z(z_prev)
        for blk in self.blocks:
            if self.grad_ckpt and self.training:
                h, z = checkpoint(blk, h, z, NB, ctx["rev_pos"], ctx["mutual"], ctx["tri"], use_reentrant=False)
            else:
                h, z = blk(h, z, NB, ctx["rev_pos"], ctx["mutual"], ctx["tri"])
        return h, z

    def forward(self, ctx, NB, n_recycle=None):
        n_rec = self.recycles if n_recycle is None else n_recycle
        h_prev = z_prev = None
        if n_rec > 0:
            with torch.no_grad():
                for _ in range(n_rec):
                    h_prev, z_prev = self.trunk(ctx, NB, h_prev, z_prev)
                    h_prev, z_prev = h_prev.detach(), z_prev.detach()
        h, z = self.trunk(ctx, NB, h_prev, z_prev)
        logit = self.edge_head(z).squeeze(-1).float()                  # (N,K) directed
        mutual, rev_pos = ctx["mutual"], ctx["rev_pos"]
        logit_rev = logit[NB, rev_pos]
        sym = torch.where(mutual, 0.5 * (logit + logit_rev), logit)
        f = self.pi_head(h).squeeze(-1).float()
        return {"edge_logit": sym, "pi_f": f}


# ------------------------------------------------------------------------- losses
def scatter_mean(x, idx, B):
    s = torch.zeros(B, device=x.device, dtype=x.dtype).index_add_(0, idx, x)
    c = torch.zeros(B, device=x.device, dtype=x.dtype).index_add_(0, idx, torch.ones_like(x))
    return s / c.clamp_min(1)


def compute_losses(out, b, w_bce=0.5, w_pi=1.0):
    logit = out["edge_logit"]
    y = b["y"].float()
    K = y.shape[1]
    has = y.sum(1) > 0
    logp = F.log_softmax(logit, 1)
    ce = -((y / y.sum(1, keepdim=True).clamp_min(1)) * logp).sum(1)[has].mean()
    bce = F.binary_cross_entropy_with_logits(logit, y, pos_weight=torch.tensor((K - 2) / 2, device=y.device))
    loss = ce + w_bce * bce
    logs = {"ce": ce.item(), "bce": bce.item()}
    if "pi_t" in b and w_pi > 0:
        s, bidx, B = b["s"], b["bidx"], b["B"]
        pi_hat, pi_tgt = s * out["pi_f"], s * b["pi_t"]
        d = (pi_hat - scatter_mean(pi_hat, bidx, B)[bidx]) - (pi_tgt - scatter_mean(pi_tgt, bidx, B)[bidx])
        mse = (d / s).pow(2).mean()
        loss = loss + w_pi * mse
        logs["pi_mse"] = mse.item()
    logs["loss"] = loss.item()
    return loss, logs


# ---------------------------------------------------------------------- inference
def autocast_dtype(device):
    if device.type == "cuda" and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return None


@torch.no_grad()
def predict(model, xy, K, device, k_out=5, n_recycle=None):
    """Return LKH-ready candidates (n,k_out), alpha ranks, integer pi and timings."""
    t0 = time.perf_counter()
    prep = prepare_instance(xy, K)
    b = to_device(collate([prep]), device)
    t1 = time.perf_counter()
    ctx = graph_context(b, k_tri=model.cfg.get("k_tri"))
    dt = autocast_dtype(device)
    model.eval()
    with torch.autocast(device_type=device.type, dtype=dt, enabled=dt is not None):
        out = model(ctx, b["NB"], n_recycle=n_recycle)
    logit = out["edge_logit"].float()
    top = logit.topk(k_out, dim=1).indices
    cand = torch.gather(b["NB"], 1, top).cpu().numpy()
    pi_hat = (b["s"] * out["pi_f"]).double()
    pi_hat = pi_hat - pi_hat.mean()
    pi_int = torch.round(pi_hat * PRECISION * prep["span"]).long().cpu().numpy()
    if device.type == "cuda":
        torch.cuda.synchronize()
    t2 = time.perf_counter()
    alpha = np.tile(np.arange(k_out, dtype=np.int64) * 100, (len(cand), 1))
    return {"cand": cand, "alpha": alpha, "pi": pi_int, "logit": logit.cpu().numpy(),
            "NB": prep["NB"], "t_prep": t1 - t0, "t_model": t2 - t1, "t_total": t2 - t0}


# =============================================================================== TourFold
# The ensemble of cheap local-optimum tours plays the role of AlphaFold's MSA:
#   column attention  = for one node, compare how the M tours connect it
#   tour convolution  = context along each tour (successor + predecessor, shared weights)
#   row attention     = for one tour and node, look at the node's kNN neighbours, biased by the pair rep
#   outer-product mean= co-variation across tours  ->  pair (edge) representation
# followed by the pair stack with triangle updates (Block above).

class MSAColumnAttention(nn.Module):
    def __init__(self, dm, heads=4):
        super().__init__()
        self.h, self.dh = heads, max(4, dm // heads)
        self.ln = nn.LayerNorm(dm)
        self.qkv = nn.Linear(dm, 3 * self.h * self.dh)
        self.g, self.o = nn.Linear(dm, self.h * self.dh), nn.Linear(self.h * self.dh, dm)

    def forward(self, msa):
        M, N, _ = msa.shape
        x = self.ln(msa)
        q, k, v = self.qkv(x).view(M, N, 3, self.h, self.dh).unbind(2)
        logit = torch.einsum("mnhd,lnhd->nhml", q, k) / math.sqrt(self.dh)
        att = logit.float().softmax(-1).to(v.dtype)
        out = torch.einsum("nhml,lnhd->mnhd", att, v).reshape(M, N, -1)
        return self.o(torch.sigmoid(self.g(x)) * out)


class MSATourConv(nn.Module):
    """msa[m,i] += f(msa[m,i], msa[m,succ_m(i)] + msa[m,pred_m(i)]) - direction-free context along each tour."""

    def __init__(self, dm):
        super().__init__()
        self.ln = nn.LayerNorm(dm)
        self.self_w, self.nbr_w = nn.Linear(dm, dm), nn.Linear(dm, dm)
        self.o = nn.Linear(dm, dm)

    def forward(self, msa, succ, pred):
        M, N, dm = msa.shape
        x = self.ln(msa)
        nb = torch.gather(x, 1, succ[..., None].expand(M, N, dm)) + torch.gather(x, 1, pred[..., None].expand(M, N, dm))
        return self.o(F.gelu(self.self_w(x) + self.nbr_w(nb)))


class MSARowAttention(nn.Module):
    """For tour m and node i: attention over i's kNN neighbours j, biased by the pair representation z_ij."""

    def __init__(self, dm, de, heads=4):
        super().__init__()
        self.h, self.dh = heads, max(4, dm // heads)
        self.ln, self.lnz = nn.LayerNorm(dm), nn.LayerNorm(de)
        self.q, self.kv = nn.Linear(dm, self.h * self.dh), nn.Linear(dm, 2 * self.h * self.dh)
        self.bias = nn.Linear(de, heads)
        self.g, self.o = nn.Linear(dm, self.h * self.dh), nn.Linear(self.h * self.dh, dm)

    def forward(self, msa, z, NB):
        M, N, _ = msa.shape
        K = NB.shape[1]
        x = self.ln(msa)
        q = self.q(x).view(M, N, self.h, self.dh)
        k, v = self.kv(x)[:, NB].split(self.h * self.dh, -1)               # (M,N,K,h*dh)
        k, v = k.view(M, N, K, self.h, self.dh), v.view(M, N, K, self.h, self.dh)
        logit = torch.einsum("mnhd,mnkhd->mnhk", q, k) / math.sqrt(self.dh)
        logit = logit + self.bias(self.lnz(z)).permute(0, 2, 1)[None]
        att = logit.float().softmax(-1).to(v.dtype)
        out = torch.einsum("mnhk,mnkhd->mnhd", att, v).reshape(M, N, -1)
        return self.o(torch.sigmoid(self.g(x)) * out)


class OuterProductMean(nn.Module):
    """z_ij += W * mean_m (a[m,i] (x) b[m,j]) on kNN edges: co-variation of how the tours treat i and j."""

    def __init__(self, dm, de, c=8):
        super().__init__()
        self.c = c
        self.ln = nn.LayerNorm(dm)
        self.a, self.b = nn.Linear(dm, c), nn.Linear(dm, c)
        self.o = nn.Linear(c * c, de)

    def forward(self, msa, NB):
        M, N, _ = msa.shape
        K = NB.shape[1]
        x = self.ln(msa)
        a, b = self.a(x), self.b(x)[:, NB]                                     # (M,N,c), (M,N,K,c)
        opm = torch.einsum("mnc,mnkd->nkcd", a.float(), b.float()) / M
        return self.o(opm.reshape(N, K, self.c * self.c).to(x.dtype))


class EvoBlock(nn.Module):
    def __init__(self, dn, de, dm, dtri, heads, tri_attn, opm_c=8):
        super().__init__()
        self.col = MSAColumnAttention(dm, heads)
        self.conv = MSATourConv(dm)
        self.row = MSARowAttention(dm, de, heads)
        self.msa_tr = Transition(dm)
        self.opm = OuterProductMean(dm, de, opm_c)
        self.pair = Block(dn, de, dtri, heads, tri_attn)

    def forward(self, msa, h, z, NB, rev_pos, mutual, tri, succ, pred):
        msa = msa + self.col(msa)
        msa = msa + self.conv(msa, succ, pred)
        msa = msa + self.row(msa, z, NB)
        msa = msa + self.msa_tr(msa)
        z = z + self.opm(msa, NB)
        h, z = self.pair(h, z, NB, rev_pos, mutual, tri)
        return msa, h, z


class TourFoldNet(nn.Module):
    """use_ensemble=False gives the single-instance ablation (same pair stack, no tour alignment)."""

    def __init__(self, dn=128, de=64, dm=32, dtri=32, msa_layers=2, pair_layers=6, heads=4,
                 tri_attn=False, use_ensemble=True, k_tri=12, K=16, opm_c=8, grad_ckpt=False,
                 node_in=5, edge_in=9, msa_in=5, graph_extra=0):
        super().__init__()
        self.cfg = dict(dn=dn, de=de, dm=dm, dtri=dtri, msa_layers=msa_layers, pair_layers=pair_layers,
                        heads=heads, tri_attn=tri_attn, use_ensemble=use_ensemble, k_tri=k_tri, K=K,
                        opm_c=opm_c, grad_ckpt=grad_ckpt, node_in=node_in, edge_in=edge_in, msa_in=msa_in,
                        graph_extra=int(graph_extra if use_ensemble else 0))
        self.use_ensemble = use_ensemble
        self.node_in = nn.Sequential(nn.Linear(node_in, dn), nn.GELU(), nn.Linear(dn, dn))
        e_in = edge_in + (2 if use_ensemble else 0) + (1 if (use_ensemble and graph_extra) else 0)
        self.edge_in = nn.Sequential(nn.Linear(e_in, de), nn.GELU(),
                                     nn.Linear(de, de))
        if use_ensemble:
            self.msa_in = nn.Sequential(nn.Linear(msa_in, dm), nn.GELU(), nn.Linear(dm, dm))
            self.pos_emb = nn.Embedding(K + 1, dm)
            self.evo = nn.ModuleList(EvoBlock(dn, de, dm, dtri, heads, tri_attn, opm_c) for _ in range(msa_layers))
        self.blocks = nn.ModuleList(Block(dn, de, dtri, heads, tri_attn) for _ in range(pair_layers))
        self.edge_head = nn.Sequential(nn.LayerNorm(de), nn.Linear(de, de), nn.GELU(), nn.Linear(de, 1))
        self.pi_head = nn.Sequential(nn.LayerNorm(dn), nn.Linear(dn, dn), nn.GELU(), nn.Linear(dn, 1))
        self.conf_head = nn.Sequential(nn.LayerNorm(dn), nn.Linear(dn, dn), nn.GELU(), nn.Linear(dn, 1))
        self.grad_ckpt = grad_ckpt

    def _run(self, fn, *args):
        if self.grad_ckpt and self.training:
            return checkpoint(fn, *args, use_reentrant=False)
        return fn(*args)

    def forward(self, ctx, b, n_recycle=None):
        NB = b["NB"]
        h = self.node_in(ctx["node_feat"])
        z = self.edge_in(ctx["edge_feat"])
        if self.use_ensemble:
            msa = self.msa_in(b["msa_f"]) + self.pos_emb(b["msa_ps"]) + self.pos_emb(b["msa_pp"])
            for blk in self.evo:
                msa, h, z = self._run(blk, msa, h, z, NB, ctx["rev_pos"], ctx["mutual"], ctx["tri"],
                                      b["ens_succ"], b["ens_pred"])
        for blk in self.blocks:
            h, z = self._run(blk, h, z, NB, ctx["rev_pos"], ctx["mutual"], ctx["tri"])
        logit = self.edge_head(z).squeeze(-1).float()
        mutual, rev_pos = ctx["mutual"], ctx["rev_pos"]
        sym = torch.where(mutual, 0.5 * (logit + logit[NB, rev_pos]), logit)
        return {"edge_logit": sym, "pi_f": self.pi_head(h).squeeze(-1).float(),
                "conf_logit": self.conf_head(h).squeeze(-1).float()}


def tourfold_losses(out, b, w_bce=0.5, w_pi=1.0, w_conf=0.2):
    loss, logs = compute_losses(out, b, w_bce, w_pi)
    if "conf_logit" in out and w_conf > 0:
        top2 = out["edge_logit"].detach().topk(2, dim=1).indices
        target = torch.gather(b["y"], 1, top2).all(1).float()
        lc = F.binary_cross_entropy_with_logits(out["conf_logit"], target)
        loss = loss + w_conf * lc
        logs["conf_bce"] = lc.item()
        logs["loss"] = loss.item()
    return loss, logs


def load_model(path, device):
    """Rebuild a TourFoldNet from a training checkpoint (its stored config) and load the weights."""
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = dict(ck["cfg"])
    cfg["grad_ckpt"] = False
    model = TourFoldNet(**cfg).to(device)
    model.load_state_dict(ck["model"])
    return model.eval(), ck


def rank_neighbours(NB, logit):
    """All graph neighbours of every node, best first by logit; equal logits (bf16 output) keep the graph's
    distance order, so the ranking is deterministic."""
    return torch.gather(NB, 1, torch.sort(logit, dim=1, descending=True, stable=True).indices)


@torch.no_grad()
def predict_tourfold(model, xy, K, device, ens_succ=None, ens_cost=None, k_out=5):
    """Candidates, integer pi, per-edge probabilities and per-node confidence for one instance.
    'rank' lists all K graph neighbours of each node, best first (see rank_neighbours)."""
    t0 = time.perf_counter()
    use = model.cfg.get("use_ensemble", False)
    prep = prepare_instance(xy, K, ens_succ=ens_succ if use else None, ens_cost=ens_cost if use else None,
                            extra=model.cfg.get("graph_extra", 0) if use else 0)
    b = to_device(collate([prep]), device)
    t1 = time.perf_counter()
    ctx = graph_context(b, k_tri=model.cfg.get("k_tri"))
    dt = autocast_dtype(device)
    model.eval()
    with torch.autocast(device_type=device.type, dtype=dt, enabled=dt is not None):
        out = model(ctx, b)
    logit = out["edge_logit"].float()
    top = logit.topk(k_out, dim=1).indices
    cand = torch.gather(b["NB"], 1, top).cpu().numpy()
    rank = rank_neighbours(b["NB"], logit).cpu().numpy()
    pi_hat = (b["s"] * out["pi_f"]).double()
    pi_hat = pi_hat - pi_hat.mean()
    pi_int = torch.round(pi_hat * PRECISION * prep["span"]).long().cpu().numpy()
    if device.type == "cuda":
        torch.cuda.synchronize()
    t2 = time.perf_counter()
    alpha = np.tile(np.arange(k_out, dtype=np.int64) * 100, (len(cand), 1))
    return {"cand": cand, "rank": rank, "alpha": alpha, "pi": pi_int, "prob": torch.sigmoid(logit).cpu().numpy(),
            "conf": torch.sigmoid(out["conf_logit"]).cpu().numpy(), "NB": prep["NB"], "D": prep["D"],
            "t_prep": t1 - t0, "t_model": t2 - t1, "t_total": t2 - t0}
'''}
for _name, _src in MODULES.items():
    with open(os.path.join(LOCAL, _name), "w") as _f:
        _f.write(_src)
if LOCAL not in sys.path:
    sys.path.insert(0, LOCAL)
import numpy as np, pandas as pd, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tsplab as T, tsp_model as M
importlib.reload(T); importlib.reload(M)

WORK = f"{LOCAL}/work"
LOG_PATH, STATUS_PATH = f"{R3}/log_v3.txt", f"{R3}/status_v3.json"
T0 = time.time()


def log(msg):
    line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
    print(line, flush=True)
    try:                       # a brief Google Drive hiccup must not end a many-hour run
        with open(LOG_PATH, "a") as f:
            f.write(line + "\n")
    except OSError:
        pass


def jdump(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    os.replace(tmp, path)


EST = {}   # stage -> estimated remaining seconds (rough defaults first, replaced by measured rates)


def hfmt(sec):
    return f"{sec / 3600:.1f} saat" if sec >= 3600 else f"{max(1, round(sec / 60))} dk"


def status(stage, state, **extra):
    try:
        jdump({"stage": stage, "state": state, "time": datetime.datetime.now().isoformat(timespec="seconds"),
               "elapsed_h": round((time.time() - T0) / 3600, 2),
               "remaining_h_estimate": round(sum(EST.values()) / 3600, 1), **extra}, STATUS_PATH)
    except OSError:
        pass


def progress(stage, done, total, t_start, label):
    """Log progress of a stage with a remaining-time estimate from the measured rate."""
    try:
        if done > 0:
            EST[stage] = (time.time() - t_start) / done * max(0, total - done)
        log(f"{label}: {done}/{total} | bu aşama için kalan ~{hfmt(EST.get(stage, 0))} | "
            f"tümü için kalan ~{hfmt(sum(EST.values()))}")
        status(stage, "running", done=done, of=total)
    except Exception:  # noqa: BLE001  (an estimate must never stop the run)
        pass


def csv_atomic(df, path):
    df.to_csv(path + ".tmp", index=False)
    os.replace(path + ".tmp", path)


def next_index(files, rx):
    return 1 + max([int(re.search(rx, f).group(1)) for f in files] or [-1])


def md(df, index=True):
    try:
        return df.to_markdown(index=index)
    except Exception:  # noqa: BLE001  (tabulate yoksa)
        return "```\n" + df.to_string(index=index) + "\n```"


def npz_parts(pattern):
    return [p for p in sorted(glob.glob(pattern)) if not p.endswith(".tmp.npz")]


def load_records(files):
    out = []
    for f in files:
        s = T.Shard(f)
        out.extend(s[i] for i in range(len(s)))
    return out


# ---------------------------------------------------------------- configuration (automatic)
ENV = T.env_info()
NPROC = ENV["nproc"] or 1
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = M.autocast_dtype(DEV)
if DEV.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
SCALES = {   # chosen from the number of CPU cores (LKH runs on the CPU)
    "L": dict(n_small=64, n_large=8, tsplib_max_n=20000, profile_sizes=[1000, 2000, 5000, 10000, 20000],
              n_train=10000, epochs=12, any_small=8, any_large=2, tsplib_any_max_n=3000),
    "M": dict(n_small=32, n_large=4, tsplib_max_n=10000, profile_sizes=[1000, 2000, 5000, 10000],
              n_train=5000, epochs=10, any_small=6, any_large=2, tsplib_any_max_n=2000),
    "S": dict(n_small=16, n_large=2, tsplib_max_n=5000, profile_sizes=[1000, 2000, 5000],
              n_train=3000, epochs=8, any_small=4, any_large=1, tsplib_any_max_n=1500),
    "tiny": dict(n_small=2, n_large=1, tsplib_max_n=10 ** 9, profile_sizes=[80, 120], n_train=40, epochs=2,
                 any_small=1, any_large=1, tsplib_any_max_n=10 ** 9),
}


def default_cfg(scale):
    t = TEST_MODE
    return dict(scale=scale, **SCALES[scale],
                size_small=120 if t else 1000, size_large=200 if t else 10000,
                M_ens=4 if t else 16, K=8 if t else 16, k_tri=6 if t else 12,
                ref_trials_cap=3 if t else 1000,
                ref_max_candidates=8,   # reference tours searched on 8 α-candidates, so they are not tied to α's top 5
                train_n=[40, 80] if t else [100, 500], shard_size=20 if t else 1000,
                val_instances=10 if t else 1000, max_nodes=300 if t else 7500, lr=3e-4, warmup=10 if t else 300,
                model=dict(dn=32, de=16, dm=8, dtri=8, msa_layers=1, pair_layers=1) if t else
                dict(dn=128, de=64, dm=32, dtri=32, msa_layers=2, pair_layers=6),
                budget_small=1.5 if t else 30.0, budget_large=2.0 if t else 300.0,
                budget_per_node=0.01 if t else 0.02, budget_min=1.5 if t else 10.0,
                fix_precision=0.999, num_workers=1 if t else min(4, max(1, NPROC // 2)),
                eval_part_size=max(4, NPROC), lkh_eval_part_size=max(1, NPROC // 2))


CFG_PATH = f"{ROOT}/config.json"
_saved = json.load(open(CFG_PATH)) if os.path.exists(CFG_PATH) else {}
_scale = _saved.get("scale") or ("tiny" if TEST_MODE else "L" if NPROC >= 8 else "M" if NPROC >= 4 else "S")
CFG = {**default_cfg(_scale), **_saved}
jdump(CFG, CFG_PATH)
TIMING_WORKERS = max(1, NPROC // 2)      # timed LKH runs: one per physical core
SUMMARY_PATH = f"{RES3}/summary_v3.json"
SUMMARY = json.load(open(SUMMARY_PATH)) if os.path.exists(SUMMARY_PATH) else {}
SUMMARY.update(env=ENV, config=CFG, errors={})
RESULTS = {}


def eta_defaults(sets=None):
    """Rough durations (s) of the long stages before they are measured; tuned on a Colab L4 runtime and
    replaced by measured rates while the cell runs."""
    eff = max(1.0, 0.7 * NPROC)                     # effective LKH throughput of hyper-threaded vCPUs
    t_ref = lambda n, clu=False: 60.0 * (n / 1000) ** 1.4 * (6.0 if clu else 1.0)  # noqa: E731  reference + ensemble
    if sets is None:
        sizes = [(CFG["size_small"], c) for c in (False, True) for _ in range(CFG["n_small"])] + \
                [(CFG["size_large"], c) for c in (False, True) for _ in range(CFG["n_large"])] + [(3000, False)] * 30
        any_b = (2 * CFG["any_small"] * CFG["budget_small"] + 2 * CFG["any_large"] * CFG["budget_large"]
                 + 20 * 35.0)
    else:
        sizes = [(len(d["xy"]), name.startswith("clu")) for name, v in sets.items() for d in v]
        any_b = 0.0
        for name, v in sets.items():
            if name == "tsplib":
                any_b += sum(max(CFG["budget_min"], CFG["budget_per_node"] * len(d["xy"])) for d in v
                             if len(d["xy"]) <= CFG["tsplib_any_max_n"])
            else:
                k = CFG["any_small"] if name.endswith("1k") else CFG["any_large"]
                any_b += min(k, len(v)) * (CFG["budget_small"] if name.endswith("1k") else CFG["budget_large"])
    f_tr = CFG["n_train"] / 30000 * CFG["epochs"] / 12
    return {"3-eval_ref": sum(t_ref(n, c) for n, c in sizes) / eff, "3-classic_coverage": 300.0,
            "4-train_data": CFG["n_train"] * 12.6 / eff, "5-train_tourfold": 7200.0 * f_tr,
            "5-train_single": 4300.0 * f_tr, "6-decision": 600.0,
            "7-lkh_eval": 1.15 * 7 * any_b / max(1, TIMING_WORKERS)}


log(f"TourFold v3 başlıyor | ortam {ENV} | cihaz {DEV} | ölçek {CFG['scale']} | veri {ROOT} | sonuçlar {R3}")
if DEV.type != "cuda" and not TEST_MODE:
    log("UYARI: GPU yok. Veri adımları yine de çalışır; model adımından önce durulur.")


# ---------------------------------------------------------------- 1) LKH + self-test
def stage_lkh():
    status("lkh", "running")
    if os.environ.get("TOURFOLD_LKH"):
        lkh = os.environ["TOURFOLD_LKH"]
    else:                      # the first session picks the newest LKH-3; later sessions rebuild that same version
        ver = CFG.get("lkh_version")
        have = glob.glob(f"{LOCAL}/lkh/LKH-{ver}/LKH") if ver else []
        lkh = have[0] if have else T.build_lkh(f"{LOCAL}/lkh", version=ver)
        m = re.search(r"LKH-(\d+\.\d+\.\d+)", lkh)
        if m and not ver:
            CFG["lkh_version"] = m.group(1)
            jdump(CFG, CFG_PATH)
    log(f"LKH hazır: {lkh} (sürüm {CFG.get('lkh_version', '?')})")
    return lkh


def stage_selftest(lkh):
    status("selftest", "running")
    crit, warn, info = {}, {}, {}

    def check(name, ok, detail="", critical=True):
        (crit if critical else warn)[name] = bool(ok)
        log(("OK    " if ok else "HATA  " if critical else "UYARI ") + name + (f" — {detail}" if detail and not ok else ""))

    n = 60 if TEST_MODE else 300
    xy = T.to_int_coords(T.gen_uniform(n, np.random.default_rng(2026)))
    r1 = T.lkh_solve(lkh, xy=xy, params={"SEED": 7, "MAX_TRIALS": n}, workroot=WORK, want_cands=True,
                     want_pi=True, keep_stdout=True)
    info["lkh_output_head"] = r1["stdout"].splitlines()[:15]
    ok1 = T.is_valid_tour(r1["tour"], n)
    check("LKH geçerli bir tur üretti", ok1, f"rc={r1['returncode']} {r1['stderr'][-300:]}")
    cost1 = T.tour_length(xy, r1["tour"]) if ok1 else None
    check("tur uzunluğu LKH'nin bildirdiğiyle aynı", cost1 is not None and r1["cost_min"] is not None
          and abs(cost1 - r1["cost_min"]) < 0.5, f"{cost1} / {r1['cost_min']}")
    check("LKH iyileşme izi okundu", len(r1["events"]) > 0)
    check("LKH ön-işleme süresi okundu", r1["pre_time"] is not None)
    check("LKH aday (CANDIDATE_FILE) ve ceza (PI_FILE) dosyaları okundu", "cand" in r1 and "pi" in r1)
    rel = {}
    if r1["lower_bound"] is not None and "pi" in r1:
        for prec in (T.LKH_PRECISION, 1):
            b = T.held_karp_bound(xy, r1["pi"], precision=prec)["bound"]
            rel[prec] = float(abs(b - r1["lower_bound"]) / max(1e-9, abs(r1["lower_bound"])))
    check("π birimi (PRECISION = 100) doğrulandı", bool(rel) and min(rel, key=rel.get) == T.LKH_PRECISION
          and rel[T.LKH_PRECISION] < 2e-3, str(rel), critical=False)
    if "cand" in r1 and "pi" in r1:
        cf, pf = f"{WORK}/selftest.cand", f"{WORK}/selftest.pi"
        T.write_candidates(cf, r1["cand"]["cand"], r1["cand"]["alpha"], r1["cand"]["dad"])
        T.write_pi(pf, r1["pi"])
        r2 = T.lkh_solve(lkh, xy=xy, params={"SEED": 7, "MAX_TRIALS": n}, workroot=WORK, cand_file=cf, pi_file=pf)
        check("kendi yazdığımız aday/π dosyaları LKH'ye kabul edildi", T.is_valid_tour(r2["tour"], n))
    ens = None
    try:
        ens = T.task_ensemble(lkh, WORK, M=3, seed0=5, xy=xy)
    except Exception as e:  # noqa: BLE001
        info["ensemble_error"] = repr(e)
    check("tur topluluğu üretildi (POPMUSIC adaylar, ascent yok)", ens is not None and ens["succ"].shape == (n, 3),
          info.get("ensemble_error", ""))
    fix_ok = False
    if ens is not None:
        bud, mt = (3, 3) if TEST_MODE else (10, 50)
        g = T.task_guided(lkh, WORK, budget=bud, seed=3, xy=xy, cand=ens["cand"], init_tour=ens["best"],
                          extra_params={"MAX_TRIALS": mt})
        check("başlangıç turu + dışarıdan aday dosyasıyla LKH çalıştı", g["cost_check"] is not None,
              g.get("stdout_tail", ""))
        nb, _ = T.knn(xy, CFG["K"])
        freq = T.ensemble_frequency(nb, ens["succ"])
        fixed = T.select_fixed_edges(nb, freq + 1e-3 * np.random.default_rng(0).random(freq.shape), 0.99)
        info["fixed_edges_tested"] = int(len(fixed))
        if len(fixed):
            try:
                gf = T.task_guided(lkh, WORK, budget=bud, seed=3, xy=xy, cand=ens["cand"], init_tour=ens["best"],
                                   fixed_edges=fixed, extra_params={"MAX_TRIALS": mt}, return_tour=True)
                if gf.get("tour") is not None:
                    s_, p_ = T.tour_succ_pred(gf["tour"])
                    fix_ok = bool(all(s_[a] == c or p_[a] == c for a, c in fixed))
                    info.update(fixed_cost_shift=gf.get("cost_shift"), fixed_len=gf.get("fixed_len"))
                    if fix_ok:
                        zero = gf["fixed_len"] > 0 and abs(gf["cost_shift"] - gf["fixed_len"]) < 0.5
                        log(f"BİLGİ LKH sabit kenarları {'0 uzunlukla' if zero else 'gerçek uzunluklarıyla'} sayıyor; "
                            "bildirdiği maliyetler gerçek tur uzunluğuna çevriliyor")
                        fix_ok = gf.get("cost_min") is not None and abs(gf["cost_min"] - gf["cost_check"]) < 0.5
            except Exception as e:  # noqa: BLE001
                info["fixed_error"] = repr(e)
    check("sabit kenarlar (FIXED_EDGES_SECTION) LKH turunda korundu", fix_ok,
          "kenar sabitleme varyantı atlanacak", critical=False)
    res = {"checks": crit, "warnings": warn, "info": info, "pi_unit_rel": {str(k): v for k, v in rel.items()},
           "fixed_edges_supported": fix_ok}
    jdump(res, f"{RES3}/selftest.json")
    SUMMARY["selftest"] = res
    if not all(crit.values()):
        raise RuntimeError("Öz-sınama başarısız. Drive'daki TourFold_TSP/v3/results/selftest.json ve v3/log_v3.txt dosyalarını bana gönderin.")
    return fix_ok


# ---------------------------------------------------------------- 2) LKH preprocessing profile
def stage_profile(lkh):
    path = f"{ROOT}/results/lkh_profile.csv"
    if os.path.exists(path):
        return pd.read_csv(path)
    status("profile", "running")
    conf = {"alpha+ascent": {}, "alpha, no ascent": {"SUBGRADIENT": "NO"},
            "POPMUSIC, no ascent": {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO"}}
    metas, kws = [], []
    for n in CFG["profile_sizes"]:
        xy = T.to_int_coords(T.gen_uniform(n, np.random.default_rng(n)))
        for name, prm in conf.items():
            metas.append((n, name))
            kws.append(dict(lkh_bin=lkh, xy=xy, workroot=WORK, params={**prm, "MAX_TRIALS": 1, "SEED": 1}))
    res = T.pmap(T.lkh_solve, kws, workers=TIMING_WORKERS, desc="LKH ön-işleme profili")
    df = pd.DataFrame([{"n": n, "config": c, "preprocessing_s": r.get("pre_time"), "wall_s": r.get("wall")}
                       for (n, c), r in zip(metas, res)])
    csv_atomic(df, path)
    log("LKH ön-işleme süresi (sn):\n" + df.pivot_table(index="n", columns="config", values="preprocessing_s").round(2).to_string())
    return df


# ---------------------------------------------------------------- 3) evaluation sets (+ ensembles)
def eval_instances():
    def synth(prefix, n, count, dist, seed):
        rng = np.random.default_rng(seed)
        return [dict(name=f"{prefix}_{i:03d}", xy=T.to_int_coords(T.gen_instance(n, dist, rng)), ew="EUC_2D",
                     problem_file=None, opt=np.nan) for i in range(count)]
    sets = {"uni1k": synth("uni1k", CFG["size_small"], CFG["n_small"], "uniform", 101),
            "clu1k": synth("clu1k", CFG["size_small"], CFG["n_small"], "clustered", 202),
            "uni10k": synth("uni10k", CFG["size_large"], CFG["n_large"], "uniform", 303),
            "clu10k": synth("clu10k", CFG["size_large"], CFG["n_large"], "clustered", 404)}
    tsp = []
    if TEST_MODE and os.environ.get("TOURFOLD_TEST_NO_TSPLIB") != "1":
        for i, n in enumerate([110, 130]):
            pth = f"{LOCAL}/tsplib/fake{i}.tsp"
            T.write_tsp(pth, np.random.default_rng(i).random((n, 2)) * 1000, name=f"fake{i}")
            tsp.append(dict(name=f"fake{i}", xy=T.read_tsplib(pth)["xy"], ew="EUC_2D", problem_file=pth, opt=np.nan))
    elif os.environ.get("TOURFOLD_TEST_NO_TSPLIB") != "1":
        try:
            paths = T.download_tsplib(f"{ROOT}/tsplib_raw")
        except Exception as e:  # noqa: BLE001
            log(f"TSPLIB indirilemedi ({e!r})")
            paths = {}
        if paths:
            log(f"TSPLIB: {len(paths)} örnek dosyası hazır")
        else:
            log("TSPLIB sunucusu şu an yanıt vermiyor; TSPLIB kümesi bu çalıştırmada atlanıyor. Hücre her "
                "çalıştırıldığında yeniden dener ve bulduğunda TSPLIB sonuçlarını rapora kendiliğinden ekler. "
                f"İsterseniz TSPLIB dosyalarını (.tsp, .tsp.gz ya da ALL_tsp.tar.gz) {ROOT}/tsplib_raw klasörüne koyun.")
        for nm, path in sorted(paths.items()):
            inst = T.read_tsplib(path)
            if inst["edge_weight_type"] not in ("EUC_2D", "CEIL_2D") or not (1000 <= inst["dimension"] <= CFG["tsplib_max_n"]):
                continue
            local = f"{LOCAL}/tsplib/{nm}.tsp"
            shutil.copy(path, local)
            tsp.append(dict(name=nm, xy=inst["xy"], ew=inst["edge_weight_type"], problem_file=local,
                            opt=float(T.TSPLIB_OPT[nm])))
    sets["tsplib"] = sorted(tsp, key=lambda d: len(d["xy"]))
    return {k: v for k, v in sets.items() if v}


def stage_eval_sets(lkh, sets):
    REF, cs = {}, CFG["eval_part_size"]
    n_all = sum(len(v) for v in sets.values())
    n_done0 = n_done = 0
    t_stage = time.time()
    for name, insts in sets.items():
        pat = f"{ROOT}/eval/{name}_part*.npz"
        files = npz_parts(pat)
        done = {str(x) for f in files for x in np.atleast_1d(T.Shard(f).scal["name"])}
        todo = [d for d in insts if d["name"] not in done]     # resume by instance name, not by part number
        n_done0 += len(insts) - len(todo)
        parts = [todo[i:i + cs] for i in range(0, len(todo), cs)]
        k0 = next_index(files, r"_part(\d+)\.npz$")
        for k, part in enumerate(parts):
            path = f"{ROOT}/eval/{name}_part{k0 + k:03d}.npz"
            status("eval_sets", "running", set=name, part=k + 1, of=len(parts))
            kws = [dict(lkh_bin=lkh, workroot=WORK, seed=1, xy=d["xy"], problem_file=d["problem_file"], ew=d["ew"],
                        name=d["name"], dist=name, ensemble_M=CFG["M_ens"],
                        params={"MAX_TRIALS": min(len(d["xy"]), CFG["ref_trials_cap"]),
                                "MAX_CANDIDATES": CFG["ref_max_candidates"]}) for d in part]
            res = T.pmap(T.task_record, kws, workers=min(NPROC, len(kws)),
                         desc=f"referans tur + topluluk: {name} {k + 1}/{len(parts)}")
            for e in T.errors_of(res)[:2]:
                log(f"HATA {name}: {e['error']}")
            ok = [r for r in res if "error" not in r]
            opt = {d["name"]: d["opt"] for d in part}
            for r in ok:
                r["opt"] = float(opt[r["name"]])
            if ok:
                T.save_shard(path, ok)
            n_done += len(part)
            progress("3-eval_ref", n_done, n_all - n_done0, t_stage, f"değerlendirme örnekleri ({name})")
        files = npz_parts(pat)
        if files:
            order, recs, seen = {d["name"]: i for i, d in enumerate(insts)}, [], set()
            for r in load_records(files):
                if r["name"] in order and r["name"] not in seen:
                    seen.add(r["name"])
                    recs.append(r)
            REF[name] = sorted(recs, key=lambda r: order[r["name"]])
            log(f"değerlendirme kümesi {name}: {len(REF[name])} örnek")
    if not REF:
        raise RuntimeError("Hiçbir değerlendirme örneği üretilemedi")
    return REF


# ---------------------------------------------------------------- 3b) classic candidate sets
def stage_classic_coverage(lkh, sets, REF):
    path = f"{ROOT}/results/coverage_classic.csv"
    old = pd.read_csv(path) if os.path.exists(path) else None
    have = set(old["set"].astype(str)) if old is not None and len(old) else set()
    if old is not None and set(REF) <= have:
        return old
    status("classic_coverage", "running")
    rows = []
    for name, refs in REF.items():
        if name in have:
            continue
        by = {d["name"]: d for d in sets[name]}
        for label, spec in {"alpha, no ascent": dict(ctype="ALPHA", subgradient=False),
                            "nearest neighbour": dict(ctype="NEAREST-NEIGHBOR")}.items():
            kws = [dict(lkh_bin=lkh, workroot=WORK, xy=by[r["name"]]["xy"], problem_file=by[r["name"]]["problem_file"],
                        ew=by[r["name"]]["ew"], **spec) for r in refs]
            res = T.pmap(T.task_candidates, kws, workers=TIMING_WORKERS, desc=f"{name}: {label}")
            for r, c in zip(refs, res):
                if "error" in c:
                    continue
                cv = T.candidate_coverage(c["cand"], r["tour"], ks=[5])[5]
                rows.append(dict(set=name, method=label, inst=r["name"], n=r["n"], cov5=cv["undirected"],
                                 missing_per_1k=1000 * cv["missing"] / r["n"], build_s=c["pre_time"]))
    df = pd.DataFrame(rows)
    if old is not None and len(old):
        df = pd.concat([old, df], ignore_index=True)
    csv_atomic(df, path)
    return df


# ---------------------------------------------------------------- 4) training data (+ ensembles)
def stage_train_data(lkh):
    S, (n_lo, n_hi) = CFG["shard_size"], CFG["train_n"]
    n_shards = math.ceil(CFG["n_train"] / S)
    todo = [sh for sh in range(n_shards) if not os.path.exists(f"{ROOT}/train/shard_{sh:04d}.npz")]
    t_stage = time.time()
    for j, sh in enumerate(todo):
        path = f"{ROOT}/train/shard_{sh:04d}.npz"
        status("train_data", "running", shard=sh + 1, of=n_shards)
        kws = []
        for i in range(S):
            seed = 1_000_000 + sh * S + i
            rng = np.random.default_rng(seed)
            n = int(rng.integers(n_lo, n_hi + 1))
            dist = "uniform" if rng.random() < 0.5 else "clustered"
            kws.append(dict(lkh_bin=lkh, workroot=WORK, seed=seed, dist=dist, ensemble_M=CFG["M_ens"],
                            xy=T.to_int_coords(T.gen_instance(n, dist, rng)),
                            params={"MAX_CANDIDATES": CFG["ref_max_candidates"]}))
        res = T.pmap(T.task_record, kws, workers=NPROC, desc=f"eğitim verisi {sh + 1}/{n_shards}")
        errs = T.errors_of(res)
        if errs:
            log(f"eğitim verisi {sh + 1}: {len(errs)} hata; örnek: {errs[0]['error']}")
        ok = [r for r in res if "error" not in r]
        if len(ok) < S // 2:
            raise RuntimeError(f"eğitim verisi parçası {sh + 1}: örneklerin yarısından fazlası başarısız")
        T.save_shard(path, ok)
        progress("4-train_data", j + 1, len(todo), t_stage, "eğitim verisi parçaları")
    return npz_parts(f"{ROOT}/train/shard_*.npz")


# ---------------------------------------------------------------- v1/v2 training helpers (only used if the v2 checkpoint is missing)


class NodeBatches(torch.utils.data.Sampler):
    """Batches of whole instances holding at most max_nodes nodes in total (activation memory grows with the
    node count: about 1.2 GB per 1000 nodes in bf16). Reshuffled every epoch, deterministic per epoch (resume)."""

    def __init__(self, sizes, max_nodes, seed=0):
        self.sizes, self.max_nodes, self.seed, self.epoch = np.asarray(sizes), int(max_nodes), seed, 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def batches(self, epoch):
        order = np.random.default_rng(self.seed + epoch).permutation(len(self.sizes))
        out, cur, tot = [], [], 0
        for j in order:
            s = int(self.sizes[j])
            if cur and tot + s > self.max_nodes:
                out.append(cur)
                cur, tot = [], 0
            cur.append(int(j))
            tot += s
        if cur:
            out.append(cur)
        return out

    def __iter__(self):
        return iter(self.batches(self.epoch))

    def __len__(self):
        return len(self.batches(self.epoch))


# ---------------------------------------------------------------- run everything
def run(name, fn, *a, critical=True, **kw):
    t0 = time.time()
    log(f"» {name}")
    try:
        return fn(*a, **kw)
    except Exception as e:  # noqa: BLE001
        SUMMARY["errors"][name] = repr(e)
        log(f"HATA ({name}): {e!r}\n{traceback.format_exc()}")
        if critical:
            raise
        return None
    finally:
        sm = SUMMARY.setdefault("stage_minutes", {})
        sm[name] = round(sm.get(name, 0.0) + (time.time() - t0) / 60, 1)
        EST[name] = 0.0


# ================================================================ TourFold v2 (model code, reused)
C2_PATH = f"{R2}/config_v2.json"
_c2_saved = json.load(open(C2_PATH)) if os.path.exists(C2_PATH) else {}
_t = TEST_MODE
C2 = {**dict(graph_extra=2 if _t else 4,                  # long ensemble edges per node (replace farthest kNN slots)
             m_choices=[2, 4] if _t else [4, 8, 12, 16],  # ensemble sizes seen in training (MSA subsampling)
             m_small=2 if _t else 8, m_full=CFG["M_ens"],
             epochs=2 if _t else 12, lr=3e-4, warmup=10 if _t else 300, fix_precision=0.999,
             b1k=1.0 if _t else 5.0, b10k=1.5 if _t else 120.0, b20k=1.5 if _t else 300.0,
             tsplib_per_node=0.01 if _t else 0.02, tsplib_min=1.0 if _t else 10.0, tsplib_min_n=0 if _t else 3000,
             any_1k=1 if _t else 8, any_10k=1 if _t else 8, n20k=1 if _t else 4, size20k=150 if _t else 20000,
             part_size=max(1, NPROC // 2)), **_c2_saved}
if not os.path.exists(C2_PATH):      # v3 never rewrites the v2 run's files
    os.makedirs(R2, exist_ok=True)
    jdump(C2, C2_PATH)
MS, MF = int(C2["m_small"]), int(C2["m_full"])
NAME_TF2 = "TourFold-v2"


def succ_to_tour(succ_col):
    n = len(succ_col)
    t = np.empty(n, dtype=np.int64)
    t[0] = 0
    for k in range(1, n):
        t[k] = succ_col[t[k - 1]]
    return t


class TrainSetV2(torch.utils.data.Dataset):
    """Training instances with a random subset of M ensemble tours (AlphaFold-style MSA subsampling) on the
    extended graph. Keys are (index, M, seed) tuples produced by NodeBatchesM."""

    def __init__(self, shards, idx):
        self.shards, self.idx = shards, idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, key):
        j, m, seed = key
        si, i = self.idx[j]
        r = self.shards[si][i]
        Mt = r["ens_succ"].shape[1]
        sub = np.sort(np.random.default_rng(seed).choice(Mt, size=min(int(m), Mt), replace=False))
        return M.prepare_instance(r["xy"], CFG["K"], tour=r["tour"], pi=r["pi"], ens_succ=r["ens_succ"][:, sub],
                                  ens_cost=np.asarray(r["ens_cost"])[sub], extra=C2["graph_extra"])


class NodeBatchesM(NodeBatches):
    """NodeBatches with one ensemble size per batch and a per-item seed for the tour subset."""

    def __init__(self, sizes, max_nodes, m_choices, seed=0):
        super().__init__(sizes, max_nodes, seed)
        self.m_choices = [int(m) for m in m_choices]

    def batches(self, epoch):
        rng = np.random.default_rng((self.seed, epoch, 7))
        out = []
        for b in super().batches(epoch):
            m = int(rng.choice(self.m_choices))
            out.append([(j, m, int(rng.integers(2 ** 31))) for j in b])
        return out


def predict_many(model, recs, m=None, bs=8, big=3000):
    """Edge probabilities, graph and top-5 candidates for each record. Ensemble models read the first m stored
    tours (all if m is None) on their own graph (kNN, or kNN + ensemble edges for v2)."""
    use = model.cfg.get("use_ensemble", False)
    ex = model.cfg.get("graph_extra", 0) if use else 0

    def ens(r):
        if not use:
            return None, None
        k = r["ens_succ"].shape[1] if m is None else int(m)
        return r["ens_succ"][:, :k], np.asarray(r["ens_cost"])[:k]
    outs = [None] * len(recs)
    small = [i for i, r in enumerate(recs) if r["n"] <= big]
    model.eval()
    for s in range(0, len(small), bs):
        ids = small[s:s + bs]
        preps = []
        for i in ids:
            es, ec = ens(recs[i])
            preps.append(M.prepare_instance(recs[i]["xy"], CFG["K"], ens_succ=es, ens_cost=ec, extra=ex))
        b = M.to_device(M.collate(preps), DEV)
        ctx = M.graph_context(b, k_tri=model.cfg.get("k_tri"))
        with torch.no_grad(), torch.autocast(device_type=DEV.type, dtype=AMP, enabled=AMP is not None):
            out = model(ctx, b)
        logit = out["edge_logit"].float()
        top = torch.gather(b["NB"], 1, logit.topk(5, dim=1).indices).cpu().numpy()
        prob = torch.sigmoid(logit).cpu().numpy()
        off = b["offsets"].cpu().numpy()
        for j, (i, p) in enumerate(zip(ids, preps)):
            a, e = off[j], off[j + 1]
            outs[i] = {"NB": p["NB"], "prob": prob[a:e], "cand": top[a:e] - a}
    for i, r in enumerate(recs):
        if outs[i] is None:
            es, ec = ens(r)
            pr = M.predict_tourfold(model, r["xy"], CFG["K"], DEV, ens_succ=es, ens_cost=ec)
            outs[i] = {"NB": pr["NB"], "prob": pr["prob"], "cand": pr["cand"]}
    return outs


def missing_per_1k(preds, recs):
    return float(np.mean([1000 * T.candidate_coverage(p["cand"], r["tour"], ks=[5])[5]["missing"] / r["n"]
                          for p, r in zip(preds, recs)]))


# ---------------------------------------------------------------- 6) train TourFold v2
def stage_train_v2(shard_paths):
    run_dir = f"{R2}/runs/{NAME_TF2}"
    os.makedirs(run_dir, exist_ok=True)
    ckpt_path = f"{run_dir}/ckpt.pt"
    shards = [T.Shard(p) for p in shard_paths]
    idx = [(si, i) for si, sh in enumerate(shards) for i in range(len(sh))]
    last = len(shards) - 1
    val_idx = [x for x in idx if x[0] == last][: CFG["val_instances"]]
    train_idx = [x for x in idx if x[0] != last] or idx
    VAL = [shards[si][i] for si, i in val_idx]
    model = M.TourFoldNet(**CFG["model"], use_ensemble=True, K=CFG["K"], k_tri=CFG["k_tri"],
                          graph_extra=C2["graph_extra"]).to(DEV)
    if DEV.type == "cuda":
        need = CFG["max_nodes"] / 1000 * (1.3 if AMP is not None else 2.2) + 1.5
        have = torch.cuda.get_device_properties(0).total_memory / 1e9
        if need > 0.9 * have:
            model.grad_ckpt = True
            log(f"{NAME_TF2}: tahmini bellek {need:.0f} GB > GPU {have:.0f} GB; gradient checkpointing açık")
    opt = torch.optim.AdamW(model.parameters(), lr=C2["lr"], weight_decay=0.01)
    sizes = [int(shards[si].off[i + 1] - shards[si].off[i]) for si, i in train_idx]
    sampler = NodeBatchesM(sizes, CFG["max_nodes"], C2["m_choices"], seed=23)
    dl_kw = dict(batch_sampler=sampler, collate_fn=M.collate, num_workers=CFG["num_workers"])
    if CFG["num_workers"] > 0:
        dl_kw.update(persistent_workers=True, multiprocessing_context="fork")
    dl = torch.utils.data.DataLoader(TrainSetV2(shards, train_idx), **dl_kw)
    E = int(C2["epochs"])
    total = max(1, sum(len(sampler.batches(e)) for e in range(E)))
    warm = C2["warmup"]
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    hist, start = [], 0
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEV, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        model.grad_ckpt = model.grad_ckpt or bool(ck.get("grad_ckpt", False))
        hist, start = ck["hist"], ck["epoch"] + 1
        if start < E:
            log(f"{NAME_TF2}: kontrol noktasından devam (epoch {start + 1}/{E})")
    if start < E:
        n_par = sum(p.numel() for p in model.parameters())
        log(f"{NAME_TF2}: {n_par / 1e6:.2f} M parametre | {len(train_idx)} eğitim, {len(VAL)} doğrulama örneği | "
            f"topluluk boyu {C2['m_choices']} | grafiğe düğüm başına en çok {C2['graph_extra']} topluluk kenarı | "
            f"~{len(sampler)} adım/epoch")
    miss_alpha = float(np.mean([1000 * T.candidate_coverage(r["cand"], r["tour"], ks=[5])[5]["missing"] / r["n"]
                                for r in VAL]))
    for epoch in range(start, E):
        status("6-train_v2", "running", epoch=epoch + 1, of=E)
        sampler.set_epoch(epoch)
        model.train()
        t0, agg, nb_, skipped = time.time(), {}, 0, 0
        for step, b in enumerate(dl):
            b = M.to_device(b, DEV)
            try:
                ctx = M.graph_context(b, k_tri=CFG["k_tri"])
                with torch.autocast(device_type=DEV.type, dtype=AMP, enabled=AMP is not None):
                    out = model(ctx, b)
                loss, lg = M.tourfold_losses(out, b)
                opt.zero_grad(set_to_none=True)
                if not torch.isfinite(loss):
                    skipped += 1
                    continue
                loss.backward()
            except torch.cuda.OutOfMemoryError:
                b = ctx = out = loss = None
                opt.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                if not model.grad_ckpt:
                    model.grad_ckpt = True
                    log(f"{NAME_TF2}: GPU belleği yetmedi; gradient checkpointing açıldı")
                else:
                    skipped += 1
                continue
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            for k, v in lg.items():
                agg[k] = agg.get(k, 0.0) + v
            nb_ += 1
            if epoch == start and step == 20:
                mem = f"tepe GPU belleği {torch.cuda.max_memory_allocated() / 1e9:.1f} GB, " if DEV.type == "cuda" else ""
                log(f"{NAME_TF2}: {mem}tahmini {(time.time() - t0) / 21 * len(dl) / 60:.0f} dk/epoch")
        val = {m: missing_per_1k(predict_many(model, VAL, m=m), VAL) for m in sorted({MS, MF})}
        rec = {"epoch": epoch, "time_s": time.time() - t0, "oom_skipped": skipped,
               **{f"train_{k}": v / max(1, nb_) for k, v in agg.items()},
               **{f"val_missing_per_1k_M{m}": v for m, v in val.items()}, "val_missing_per_1k_alpha": miss_alpha}
        hist.append(rec)
        tmp = ckpt_path + ".tmp"
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "hist": hist,
                    "epoch": epoch, "cfg": model.cfg, "grad_ckpt": model.grad_ckpt}, tmp)
        os.replace(tmp, ckpt_path)
        EST["6-train_v2"] = rec["time_s"] * (E - epoch - 1)
        log(f"{NAME_TF2} epoch {epoch + 1}/{E}: kayıp {rec.get('train_loss', float('nan')):.4f} | eksik tur kenarı/1000 "
            f"(ilk 5 aday): " + ", ".join(f"M={m} {v:.2f}" for m, v in val.items()) + f", α+ascent {miss_alpha:.2f} | "
            f"{rec['time_s'] / 60:.1f} dk" + (f" | {skipped} parti atlandı" if skipped else "")
            + f" | bu model için kalan ~{hfmt(EST['6-train_v2'])}, tümü için ~{hfmt(sum(EST.values()))}")
        status("6-train_v2", "running", epoch=epoch + 1, of=E)
    SUMMARY["training_v2"] = hist
    return model.eval()


# ---------------------------------------------------------------- LKH comparison instances (as in v2)


def lkh_eval_selection(REF, sets):
    sel = []
    for name, key, bkey in (("uni1k", "any_1k", "b1k"), ("clu1k", "any_1k", "b1k"),
                            ("uni10k", "any_10k", "b10k"), ("clu10k", "any_10k", "b10k")):
        by = {d["name"]: d for d in sets.get(name, [])}
        for r in REF.get(name, [])[: C2[key]]:
            d = by[r["name"]]
            sel.append((name, dict(name=r["name"], xy=d["xy"], ew=d["ew"], problem_file=d["problem_file"],
                                   opt=float(r["opt"]), ref_cost=float(r["cost"])), float(C2[bkey])))
    by = {d["name"]: d for d in sets.get("tsplib", [])}
    for r in REF.get("tsplib", []):
        if r["n"] >= C2["tsplib_min_n"]:
            d = by[r["name"]]
            sel.append(("tsplib", dict(name=r["name"], xy=d["xy"], ew=d["ew"], problem_file=d["problem_file"],
                                       opt=float(r["opt"]), ref_cost=float(r["cost"])),
                        float(max(C2["tsplib_min"], C2["tsplib_per_node"] * r["n"]))))
    for dist, seed, pre in (("uniform", 505, "uni"), ("clustered", 606, "clu")):
        rng = np.random.default_rng(seed)
        for i in range(int(C2["n20k"])):
            xy = T.to_int_coords(T.gen_instance(int(C2["size20k"]), dist, rng))
            sel.append((f"{pre}20k", dict(name=f"{pre}20k_{i:03d}", xy=xy, ew="EUC_2D", problem_file=None,
                                          opt=float("nan"), ref_cost=float("inf")), float(C2["b20k"])))
    return sorted(sel, key=lambda x: x[2])


# ================================================================ TourFold v3
# No new training: the TourFold-v2 model of the v2 run proposes candidates. Mixed lists keep TourFold's first
# entries and fill up from POPMUSIC or the nearest neighbours; a 4-tour ensemble saves part of the ensemble cost; a
# control run (POPMUSIC candidates + the ensemble's best tour as start) separates the start tour's effect from the
# candidates' effect. Ensemble seed and LKH seed are those of v2, so the 8-tour ensembles equal the first 8 tours of
# the v2 ensembles and three configurations repeat v2 runs (a check of run-to-run timing noise).
C3_PATH = f"{R3}/config_v3.json"
_c3_saved = json.load(open(C3_PATH)) if os.path.exists(C3_PATH) else {}
_mb = min(8, int(CFG["M_ens"]))
C3 = {**dict(m_big=_mb, m_small=max(2, _mb // 2),   # ensemble of 8 tours, and its first 4 tours
             k_tf=3, k=5,                           # mixed lists: TourFold's first 3 + 2 entries from another list
             k_union=8, k_near=8, ens_seed=777, lkh_seed=101, part_size=max(1, NPROC // 2)), **_c3_saved}
jdump(C3, C3_PATH)
MB, MSM, KT, KC = int(C3["m_big"]), int(C3["m_small"]), int(C3["k_tf"]), int(C3["k"])
assert MSM < MB and 0 < KT < KC
CONF = {"pop": "LKH-3 POPMUSIC",
        "ctrl": f"Kontrol: POPMUSIC + topluluk({MB}) başlangıcı",
        "cnt": f"sayım+ ({MB})",
        "tf5": f"TourFold-v2({MB}) ilk {KC}",
        "tfpop": f"TourFold-v2({MB}) {KT}+POPMUSIC {KC - KT}",
        "tfnear": f"TourFold-v2({MB}) {KT}+yakın {KC - KT}",
        "tfuni": f"TourFold-v2({MB}) ∪ POPMUSIC",
        "tf4pop": f"TourFold-v2({MSM}) {KT}+POPMUSIC {KC - KT}"}
COV_NAMES = {"tf5": CONF["tf5"], "tf4_5": f"TourFold-v2({MSM}) ilk {KC}", "tfpop": CONF["tfpop"],
             "tf4pop": CONF["tf4pop"], "tfnear": CONF["tfnear"], "tfuni": CONF["tfuni"], "cnt": CONF["cnt"],
             "pop": "POPMUSIC", "alpha": "α+ascent (LKH-3 varsayılan)", "near5": f"en yakın {KC} komşu"}
CLASSES = ["1k", "10k", "TSPLIB", "20k"]
LKH_RATE = 1.19 * 6.63 / max(1, TIMING_WORKERS)   # wall s per budget s: 8 runs/instance + ensembles (v2 run, L4)


def size_class(s):
    return "TSPLIB" if s == "tsplib" else s[3:] if s[:3] in ("uni", "clu") else s


def eta_v3():
    """Rough durations (s) of the remaining stages before they are measured (calibrated on the v2 run)."""
    tsp_b = 0.0 if TEST_MODE else 2071.0                     # TSPLIB budgets of the v2 run (n >= 3000)
    b = 2 * C2["any_1k"] * C2["b1k"] + 2 * C2["any_10k"] * C2["b10k"] + 2 * C2["n20k"] * C2["b20k"] + tsp_b
    est = {"7-coverage_v3": 240.0, "8-lkh_eval_v3": LKH_RATE * b}
    if not npz_parts(f"{ROOT}/eval/*_part*.npz"):
        est["3-eval_ref"] = eta_defaults()["3-eval_ref"]
    if len(npz_parts(f"{ROOT}/train/shard_*.npz")) < math.ceil(CFG["n_train"] / CFG["shard_size"]):
        est["4-train_data"] = eta_defaults()["4-train_data"]
    if not os.path.exists(f"{R2}/runs/{NAME_TF2}/ckpt.pt"):
        est["6-train_v2"] = 8.2 * 60 * C2["epochs"] * CFG["n_train"] / 30000
    return est


EST.update(eta_v3())
try:
    _env2 = json.load(open(f"{RES2}/summary_v2.json")).get("env") or {}
except Exception:  # noqa: BLE001
    _env2 = {}
if _env2 and (_env2.get("nproc") != ENV.get("nproc") or _env2.get("gpu") != ENV.get("gpu")):
    log(f"UYARI: çalışma zamanı v2'dekinden farklı (v2: {_env2.get('nproc')} vCPU, {_env2.get('gpu')}; şimdi: "
        f"{ENV.get('nproc')} vCPU, {ENV.get('gpu')}). Süreler v2 ile birebir karşılaştırılamaz ve LKH aşaması "
        f"~{hfmt(EST['8-lkh_eval_v3'])} sürer. Mümkünse L4 GPU + yüksek RAM seçip hücreyi yeniden başlatın.")


def get_tf2(shard_paths):
    """TourFold-v2 from the v2 run's checkpoint on Drive. Training runs only if that checkpoint is missing or
    unfinished (it then resumes where it stopped)."""
    path = f"{R2}/runs/{NAME_TF2}/ckpt.pt"
    if os.path.exists(path):
        model, ck = M.load_model(path, DEV)
        done = int(ck.get("epoch", -1)) + 1
        if done >= int(C2["epochs"]):
            SUMMARY["training_v2"] = ck.get("hist", [])
            n_par = sum(p.numel() for p in model.parameters())
            log(f"{NAME_TF2} Drive'dan yüklendi: {n_par / 1e6:.2f} M parametre, {done} epoch; yeni eğitim yapılmıyor")
            return model
        h = ck.get("hist") or [{"time_s": 8.2 * 60 * CFG["n_train"] / 30000}]
        EST["6-train_v2"] = float(np.mean([x["time_s"] for x in h])) * (int(C2["epochs"]) - done)
        log(f"{NAME_TF2} kontrol noktası yarım ({done}/{C2['epochs']} epoch); eğitim kaldığı yerden sürüyor")
        del model, ck
    return stage_train_v2(shard_paths)


def load_val(shard_paths):
    """Validation instances of v1/v2 (never trained on): the first val_instances records of the last shard."""
    sh = T.Shard(shard_paths[-1])
    return [sh[i] for i in range(min(len(sh), int(CFG["val_instances"])))]


def rank_many(model, recs, m, bs=8, big=3000):
    """TourFold-v2 ranking (n, K) of every node's graph neighbours, best first, from the first m stored tours."""
    ex = model.cfg.get("graph_extra", 0)
    outs = [None] * len(recs)
    small = [i for i, r in enumerate(recs) if r["n"] <= big]
    model.eval()
    for s in range(0, len(small), bs):
        ids = small[s:s + bs]
        preps = [M.prepare_instance(recs[i]["xy"], CFG["K"], ens_succ=recs[i]["ens_succ"][:, :m],
                                    ens_cost=np.asarray(recs[i]["ens_cost"])[:m], extra=ex) for i in ids]
        b = M.to_device(M.collate(preps), DEV)
        ctx = M.graph_context(b, k_tri=model.cfg.get("k_tri"))
        with torch.no_grad(), torch.autocast(device_type=DEV.type, dtype=AMP, enabled=AMP is not None):
            out = model(ctx, b)
        rk = M.rank_neighbours(b["NB"], out["edge_logit"].float()).cpu().numpy()
        off = b["offsets"].cpu().numpy()
        for j, i in enumerate(ids):
            outs[i] = rk[off[j]:off[j + 1]] - off[j]
    for i, r in enumerate(recs):
        if outs[i] is None:
            outs[i] = M.predict_tourfold(model, r["xy"], CFG["K"], DEV, ens_succ=r["ens_succ"][:, :m],
                                         ens_cost=np.asarray(r["ens_cost"])[:m])["rank"]
    return outs


def count_plus(xy, succ):
    """'sayım+': each node's neighbours on the extended graph (kNN + ensemble edges, the graph TourFold-v2 sees),
    ranked by the share of ensemble tours that use the edge; ties by distance."""
    xy = np.asarray(xy, dtype=np.float64)
    mn = xy.min(0)
    xyn = (xy - mn) / (float((xy.max(0) - mn).max()) or 1.0)
    NB, D = M.knn_np(xyn, CFG["K"])
    NB, D, _ = M.extend_graph(xyn, NB, succ, int(C2["graph_extra"]))
    return T.frequency_candidates(NB, D, T.ensemble_frequency(NB, succ))


def build_lists(rank_b, rank_s, pop, near_fn):
    """The v3 candidate lists and the time (s) each takes to build on top of the model output.
    rank_b / rank_s: TourFold-v2 rankings (n, K) from MB / MSM tours; pop: POPMUSIC list; near_fn() -> kNN list."""
    out, dt = {}, {}

    def put(key, fn):
        t0 = time.perf_counter()
        out[key] = fn()
        dt[key] = time.perf_counter() - t0
    put("near", near_fn)
    put("tf5", lambda: rank_b[:, :KC])
    put("tfpop", lambda: T.mix_candidates(rank_b, pop, KT, KC, fill=rank_b[:, KT:]))
    put("tfnear", lambda: T.mix_candidates(rank_b, out["near"], KT, KC, fill=rank_b[:, KT:]))
    dt["tfnear"] += dt["near"]
    put("tfuni", lambda: T.mix_candidates(rank_b, pop, KC, int(C3["k_union"])))
    put("tf4_5", lambda: rank_s[:, :KC])
    put("tf4pop", lambda: T.mix_candidates(rank_s, pop, KT, KC, fill=rank_s[:, KT:]))
    return out, dt


# ---------------------------------------------------------------- 7) candidate-list quality (no LKH)
def stage_coverage_v3(tf2, VAL, REF):
    """Quality of the candidate lists against the reference tours: missing tour edges per 1000 nodes and the share
    of tour edges among each node's first 2 / 3 entries."""
    path = f"{RES3}/coverage_v3.json"
    groups = {"val": VAL, **REF}
    if os.path.exists(path):
        cached = json.load(open(path))
        if sorted(cached.get("groups", [])) == sorted(groups):
            return cached
    rows, t_stage = [], time.time()
    n_all, n_done = sum(len(v) for v in groups.values()), 0
    for gname, recs in groups.items():
        status("7-coverage_v3", "running", group=gname)
        rk = {m: rank_many(tf2, recs, m) for m in (MB, MSM)}
        for i, r in enumerate(recs):
            rid = r["name"] or f"{gname}_{i:05d}"
            sub = f"val-{r.get('dist') or '?'}" if gname == "val" else gname
            pop = np.asarray(r["pop_cand"])
            lists, _ = build_lists(rk[MB][i], rk[MSM][i], pop, lambda: T.knn(r["xy"], int(C3["k_near"]))[0])
            lists["near5"] = lists.pop("near")[:, :KC]
            lists.update(cnt=count_plus(r["xy"], r["ens_succ"][:, :MB]), pop=pop, alpha=np.asarray(r["cand"]))
            for key, cand in lists.items():
                L = int(cand.shape[1])
                cv = T.candidate_coverage(cand, r["tour"], ks=sorted({2, 3, L}))
                rows.append(dict(group=gname, sub=sub, method=COV_NAMES[key], inst=rid, n=int(r["n"]),
                                 cov2=cv[2]["undirected"], cov3=cv[3]["undirected"],
                                 missing_per_1k=1000 * cv[L]["missing"] / r["n"],
                                 list_len=float((cand >= 0).sum(1).mean())))
        n_done += len(recs)
        progress("7-coverage_v3", n_done, n_all, t_stage, f"aday listesi kalitesi ({gname})")
    df = pd.DataFrame(rows)
    csv_atomic(df, f"{RES3}/coverage_v3.csv")
    tab = df.groupby(["group", "method"])[["missing_per_1k", "cov2", "cov3", "list_len"]].mean()
    tab_sub = df[df["group"] == "val"].groupby(["sub", "method"])[["missing_per_1k"]].mean()
    log("Aday listesi kalitesi — eksik tur kenarı / 1000 düğüm (küçük = iyi):\n"
        + tab["missing_per_1k"].unstack(0).round(2).to_string())
    out = {"coverage": tab.reset_index().to_dict("records"), "val_by_dist": tab_sub.reset_index().to_dict("records"),
           "groups": list(groups)}
    jdump(out, path)
    try:
        show = [COV_NAMES[k] for k in ("tf5", "tfpop", "tfnear", "tfuni", "tf4pop", "cnt", "pop", "alpha")]
        u = tab["missing_per_1k"].unstack(0)
        fig, ax = plt.subplots(figsize=(11, 4.2))
        u.loc[[c for c in show if c in u.index]].T.plot.bar(ax=ax, rot=0)
        ax.set_yscale("symlog", linthresh=0.1)
        ax.set_xlabel("")
        ax.set_ylabel("eksik tur kenarı / 1000 düğüm"); ax.set_title("v3: aday listelerinin kalitesi (küçük = iyi)")
        ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
        plt.tight_layout(); fig.savefig(f"{R3}/figures/coverage_v3.png", dpi=130); plt.close(fig)
    except Exception as e:  # noqa: BLE001
        log(f"grafik çizilemedi: {e!r}")
    return out


# ---------------------------------------------------------------- 8) LKH time-to-quality v3 (same session)
def best_known(*run_lists):
    """Optimum (TSPLIB), otherwise the best of the reference tour and every recorded run (v2 and v3)."""
    best = {}
    for runs in run_lists:
        for x in runs:
            if np.isfinite(x["opt"]):
                best[x["inst"]] = float(x["opt"])
            else:
                cmin = min([c for c, _ in x["events"]] or [np.inf])
                best[x["inst"]] = min(best.get(x["inst"], float(x["ref_cost"])), float(cmin))
    return best


FRACS, TARGETS = (0.1, 0.25, 0.5, 1.0), (1.0, 0.5, 0.2, 0.1, 0.05)


def lkh_rows(runs, best):
    rows = []
    for x in runs:
        b = best.get(x["inst"], np.nan)
        if not np.isfinite(b):
            continue
        gaps = T.curve_abs(x["events"], b, [f * x["budget"] for f in FRACS])
        rows.append(dict(set=x["set"], inst=x["inst"], n=x["n"], config=x["config"], budget=x["budget"],
                         **{f"gap@{int(f * 100)}%B": v for f, v in zip(FRACS, gaps)},
                         **{f"time_to_{t}%": T.time_to_gap_abs(x["events"], b, t) for t in TARGETS},
                         ens_time=x.get("ens_time", 0.0) if x.get("with_ens") else 0.0))
    return pd.DataFrame(rows)


def wtl_counts(piv, c, b_):
    a_, bb = piv[c], piv[b_]
    w = int(((a_ < bb - 1e-9) | (a_.notna() & bb.isna())).sum())
    l_ = int(((a_ > bb + 1e-9) | (a_.isna() & bb.notna())).sum())
    return w, int(len(piv)) - w - l_, l_


def sign_p(w, l_):
    """Two-sided sign test (ties dropped)."""
    if w + l_ == 0:
        return float("nan")
    from scipy.stats import binomtest
    return float(binomtest(int(w), int(w + l_), 0.5).pvalue)


def pooled_wtl(df, pairs, col):
    """Win/tie/loss per size class and over all instances for (config, rival) pairs at one budget point."""
    df = df.assign(cls=df["set"].map(size_class))
    out = []
    for c, b_ in pairs:
        rec, W, Tt, L = {"yapılandırma": c, "rakip": b_}, 0, 0, 0
        for cls in CLASSES:
            piv = df[df["cls"] == cls].pivot_table(index="inst", columns="config", values=col, aggfunc="first",
                                                   dropna=False)
            if c in piv and b_ in piv:
                w, t, l_ = wtl_counts(piv, c, b_)
                rec[cls] = f"{w}/{t}/{l_}"
                W, Tt, L = W + w, Tt + t, L + l_
        rec["tümü"], rec["p (işaret testi)"] = f"{W}/{Tt}/{L}", sign_p(W, L)
        out.append(rec)
    return pd.DataFrame(out)


def load_v2_runs(sel):
    """v2 runs on the same instances (same names and sizes), for the best-known costs and the repeat check."""
    want = {d["name"]: int(len(d["xy"])) for _, d, _ in sel}
    out = []
    for f in sorted(glob.glob(f"{RES2}/lkh_eval_parts/part_*.json")):
        try:
            out += [x for x in json.load(open(f)) if want.get(x["inst"]) == int(x["n"])]
        except Exception:  # noqa: BLE001
            pass
    return out


def stage_lkh_eval_v3(lkh, tf2, REF, sets):
    path = f"{RES3}/lkh_eval_v3.json"
    part_dir = f"{RES3}/lkh_eval_parts"
    os.makedirs(part_dir, exist_ok=True)
    K = CFG["K"]
    sel = lkh_eval_selection(REF, sets)
    files = sorted(glob.glob(f"{part_dir}/part_*.json"))
    done = {x["inst"] for f in files for x in json.load(open(f))}
    todo = [x for x in sel if x[1]["name"] not in done]
    if not todo and os.path.exists(path):
        return json.load(open(path))
    b_all = sum(B for *_, B in todo)
    EST["8-lkh_eval_v3"] = LKH_RATE * b_all
    t_stage, b_done = time.time(), 0.0
    chunk = int(C3["part_size"])
    n_parts, k0 = math.ceil(len(todo) / chunk), next_index(files, r"part_(\d+)\.json$")
    if todo:   # warm-up: CUDA start-up cost must not enter the timings
        x0 = todo[0][1]["xy"]
        M.predict_tourfold(tf2, x0, K, DEV, ens_succ=np.stack([np.roll(np.arange(len(x0)), -1)] * MB, 1),
                           ens_cost=np.ones(MB))
    log(f"LKH karşılaştırması v3: {len(sel)} örnek ({len(todo)} kaldı), {n_parts} parça, {len(CONF)} yapılandırma; "
        f"topluluk ({MB} tur) oturum içinde üretilip süresi sayılır | bu aşama için tahmini ~{hfmt(EST['8-lkh_eval_v3'])}")
    gb = lambda B, o: max(0.5, B - o)   # noqa: E731
    pop_par = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO"}
    for k in range(n_parts):
        ppath = f"{part_dir}/part_{k0 + k:03d}.json"
        status("8-lkh_eval_v3", "running", part=k + 1, of=n_parts)
        part = todo[k * chunk:(k + 1) * chunk]
        ens = T.pmap(T.task_ensemble, [dict(lkh_bin=lkh, workroot=WORK, M=MB, seed0=int(C3["ens_seed"]), xy=d["xy"],
                                            problem_file=d["problem_file"], ew=d["ew"]) for _, d, _ in part],
                     workers=TIMING_WORKERS, desc=f"topluluk {k + 1}/{n_parts}", ctx="spawn")
        jobs, metas = [], []
        for (name, d, B), e in zip(part, ens):
            if "error" in e:
                log(f"topluluk hatası {d['name']}: {e['error']}")
                continue
            succ, costs, walls = e["succ"], np.asarray(e["costs"], dtype=float), np.asarray(e["walls"], dtype=float)
            t_ens = {m: float(walls[:m].sum()) for m in (MB, MSM)}
            init = {m: succ_to_tour(succ[:, int(np.argmin(costs[:m]))]) for m in (MB, MSM)}

            def evs(m):   # the ensemble's own tours: best cost so far, available after the cumulative run time
                out, best_c = [], np.inf
                for c, t in zip(costs[:m], np.cumsum(walls[:m])):
                    best_c = min(best_c, float(c))
                    out.append((best_c, float(t)))
                return out
            t0 = time.perf_counter()
            cand_cnt = count_plus(d["xy"], succ[:, :MB])
            t_cnt = time.perf_counter() - t0
            pb = M.predict_tourfold(tf2, d["xy"], K, DEV, ens_succ=succ[:, :MB], ens_cost=costs[:MB])
            psm = M.predict_tourfold(tf2, d["xy"], K, DEV, ens_succ=succ[:, :MSM], ens_cost=costs[:MSM])
            lists, dt = build_lists(pb["rank"], psm["rank"], e["cand"], lambda: T.knn(d["xy"], int(C3["k_near"]))[0])
            o_b, o_s = t_ens[MB] + pb["t_total"], t_ens[MSM] + psm["t_total"]
            spec = {"pop": (None, 0.0, 0, 0.0), "ctrl": (e["cand"], t_ens[MB], MB, 0.0),
                    "cnt": (cand_cnt, t_ens[MB] + t_cnt, MB, 0.0)}
            for key in ("tf5", "tfpop", "tfnear", "tfuni"):
                spec[key] = (lists[key], o_b + dt[key], MB, pb["t_total"])
            spec["tf4pop"] = (lists["tf4pop"], o_s + dt["tf4pop"], MSM, psm["t_total"])
            common = dict(lkh_bin=lkh, workroot=WORK, seed=int(C3["lkh_seed"]), xy=d["xy"],
                          problem_file=d["problem_file"], ew=d["ew"])
            for key, (cand, off, m_used, t_inf) in spec.items():
                if cand is None:
                    fn, kw = T.task_anytime, dict(budget=B, params=pop_par, timeout=B + 15)
                else:
                    bl = gb(B, off)
                    fn, kw = T.task_guided, dict(budget=bl, cand=cand, init_tour=init[m_used], timeout=bl + 15)
                metas.append(dict(set=name, inst=d["name"], n=int(len(d["xy"])), config=CONF[key], key=key, budget=B,
                                  offset=float(off), with_ens=m_used > 0, ens_events=evs(m_used) if m_used else [],
                                  ens_time=t_ens.get(m_used, 0.0), ens_best=float(costs[:m_used].min()) if m_used else None,
                                  t_inf=float(t_inf), list_len=None if cand is None else float((cand >= 0).sum(1).mean()),
                                  opt=float(d["opt"]), ref_cost=float(d["ref_cost"])))
                jobs.append(dict(fn=fn, kw={**common, **kw}))
        res = T.pmap(T.call_task, jobs, workers=TIMING_WORKERS, desc=f"LKH koşuları {k + 1}/{n_parts}", ctx="spawn")
        runs = []
        for m_, rr in zip(metas, res):
            if "error" in rr:
                log(f"LKH koşu hatası {m_['inst']} / {m_['config']}: {rr['error']}")
                continue
            events = m_.pop("ens_events") + T.abs_events(rr, offset=m_["offset"])
            runs.append({**m_, "events": events, "cost_min": rr.get("cost_min"), "cost_check": rr.get("cost_check"),
                         "pre_time": rr.get("pre_time"), "timed_out": rr.get("timed_out")})
        jdump(runs, ppath)
        b_done += sum(B for *_, B in part)
        if b_done > 0:
            EST["8-lkh_eval_v3"] = (time.time() - t_stage) / b_done * max(0.0, b_all - b_done)
        log(f"LKH karşılaştırması v3: {min(len(todo), (k + 1) * chunk)}/{len(todo)} örnek | bu aşama için kalan "
            f"~{hfmt(EST['8-lkh_eval_v3'])} | tümü için kalan ~{hfmt(sum(EST.values()))}")
        status("8-lkh_eval_v3", "running", part=k + 1, of=n_parts)
    runs = [x for f in sorted(glob.glob(f"{part_dir}/part_*.json")) for x in json.load(open(f))]
    if not runs:
        raise RuntimeError("LKH karşılaştırmasında hiçbir koşu tamamlanamadı")
    runs_v2 = load_v2_runs(sel)
    best = best_known(runs, runs_v2)
    df = lkh_rows(runs, best)
    csv_atomic(df, f"{RES3}/lkh_eval_v3.csv")
    gcols = [c for c in df.columns if c.startswith("gap@")]
    tcols = [c for c in df.columns if c.startswith("time_to_")]
    order = [CONF[k] for k in CONF]           # rows are written instance by instance in this order
    sets_order = list(dict.fromkeys(df["set"]))
    gap_tab = df.groupby(["set", "config"], sort=False)[gcols].mean()
    cnt = df.groupby(["set", "config"], sort=False)[["gap@10%B", "gap@25%B", "gap@100%B"]].count()
    cnt.columns = ["turlu@10%B", "turlu@25%B", "turlu@100%B"]
    gap_tab = gap_tab.join(cnt)
    gap_tab["örnek"] = df.groupby(["set", "config"], sort=False).size()
    t_tab = df.groupby(["set", "config"], sort=False)[tcols].median()
    off_tab = pd.DataFrame([dict(set=x["set"], config=x["config"], toplam=x["offset"],
                                 topluluk=x["ens_time"] if x["with_ens"] else 0.0, model=x["t_inf"]) for x in runs])
    off_tab = off_tab.groupby(["set", "config"], sort=False).mean()
    tf_keys = [k for k in CONF if k.startswith("tf")]
    pairs_pop = [(CONF[k], CONF["pop"]) for k in CONF if k != "pop"]
    pairs_ctrl = [(CONF[k], CONF["ctrl"]) for k in ["cnt"] + tf_keys]
    pairs_tf = [(CONF[k], CONF["tf5"]) for k in ("tfpop", "tfnear", "tfuni")] + [(CONF["tf4pop"], CONF["tfpop"])]
    pooled = {col: pooled_wtl(df, pairs_pop + pairs_ctrl + pairs_tf, col) for col in ("gap@25%B", "gap@50%B", "gap@100%B")}
    for col, p_ in pooled.items():
        csv_atomic(p_, f"{RES3}/lkh_eval_v3_pooled_{col.replace('@', '_').replace('%', '')}.csv")
    wtl = []
    for sname in df["set"].unique():
        sub = df[df["set"] == sname]
        for col in ("gap@25%B", "gap@100%B"):
            piv = sub.pivot_table(index="inst", columns="config", values=col, aggfunc="first", dropna=False)
            for c in [CONF[k] for k in tf_keys]:
                for b_ in [CONF[k] for k in ("pop", "ctrl", "cnt")]:
                    if c in piv and b_ in piv:
                        w, t, l_ = wtl_counts(piv, c, b_)
                        wtl.append(dict(set=sname, at=col, config=c, vs=b_, wins=w, ties=t, losses=l_))
    wtl = pd.DataFrame(wtl)
    if len(wtl):
        csv_atomic(wtl, f"{RES3}/lkh_eval_v3_wtl.csv")
    rep = []   # repeat check: the same configuration in v2 and v3, gaps against the same best-known costs
    if runs_v2:
        d2 = lkh_rows(runs_v2, best)
        pairs = [("LKH-3 POPMUSIC (ascent yok)", CONF["pop"])]
        if MS == MB:
            pairs += [(f"Topluluk({MS}) + sayım+", CONF["cnt"]), (f"TourFold-v2({MS})", CONF["tf5"])]
        for c2, c3 in pairs:
            a2, a3 = d2[d2["config"] == c2], df[df["config"] == c3]
            for sname in sets_order:
                s2, s3 = a2[a2["set"] == sname], a3[a3["set"] == sname]
                if len(s2) and len(s3):
                    rep.append({"set": sname, "yapılandırma (v3)": c3, "v2 gap@25%B": s2["gap@25%B"].mean(),
                                "v3 gap@25%B": s3["gap@25%B"].mean(), "v2 gap@100%B": s2["gap@100%B"].mean(),
                                "v3 gap@100%B": s3["gap@100%B"].mean(), "örnek": int(min(len(s2), len(s3)))})
        ens2 = {x["inst"]: x["ens_time"] for x in runs_v2 if x["config"] == f"TourFold-v2({MS})"}
        ens3 = {x["inst"]: x["ens_time"] for x in runs if x["key"] == "tf5"}
        common_i = [i for i in ens3 if i in ens2]
        if common_i and MS == MB:
            SUMMARY["ens_time_ratio_v3_v2"] = float(np.median([ens3[i] / max(1e-9, ens2[i]) for i in common_i]))
    rep = pd.DataFrame(rep)
    if len(rep):
        csv_atomic(rep, f"{RES3}/lkh_eval_v3_vs_v2.csv")
    log("LKH v3 — aynı süre bütçesinde fark (%, en iyi bilinene göre; küçük = iyi):\n" + gap_tab.round(4).to_string())
    log("LKH v3 — toplu kazanç/beraberlik/kayıp (%100 bütçe):\n" + pooled["gap@100%B"].to_string(index=False))
    out = {"gap": gap_tab.reset_index().to_dict("records"), "time_to_gap": t_tab.reset_index().to_dict("records"),
           "pooled": {k: v.to_dict("records") for k, v in pooled.items()},
           "wtl": wtl.to_dict("records") if len(wtl) else [], "repeat_v2": rep.to_dict("records") if len(rep) else [],
           "offsets": off_tab.reset_index().to_dict("records"), "runs": runs, "best": best}
    jdump(out, path)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            for sname in sets_order:
                fig, ax = plt.subplots(figsize=(8, 4.4))
                for cname in order:
                    xs = [x for x in runs if x["set"] == sname and x["config"] == cname and np.isfinite(best[x["inst"]])]
                    if not xs:
                        continue
                    Bm = float(np.mean([x["budget"] for x in xs]))
                    ts = np.linspace(0.02 * Bm, Bm, 60)
                    arr = np.array([T.curve_abs(x["events"], best[x["inst"]], ts) for x in xs], dtype=float)
                    ax.plot(ts, np.where(np.isnan(arr).any(0), np.nan, arr.mean(0)), label=cname,
                            ls="--" if cname in (CONF["pop"], CONF["ctrl"]) else "-")
                ax.set_yscale("symlog", linthresh=0.01)
                ax.set_xlabel("toplam süre (sn); eğri, her örnekte bir tur bulunduğu andan başlar")
                ax.set_ylabel("en iyi bilinene göre ortalama fark (%)")
                ax.set_title(f"v3 · {sname}: süre–kalite"); ax.legend(fontsize=7)
                plt.tight_layout(); fig.savefig(f"{R3}/figures/anytime_v3_{sname}.png", dpi=130); plt.close(fig)
    except Exception as e:  # noqa: BLE001
        log(f"grafik çizilemedi: {e!r}")
    return out


# ---------------------------------------------------------------- 9) report v3
def write_report_v3():
    L = ["# TourFold TSP v3 — otomatik rapor", "",
         f"- Tarih: {datetime.datetime.now():%Y-%m-%d %H:%M}",
         f"- Ortam: {ENV}",
         f"- Model: TourFold-v2 (v2 çalıştırmasının kontrol noktası; v3'te yeni eğitim yok). Topluluk {MB} tur; "
         f"'{CONF['tf4pop']}' aynı topluluğun yalnız ilk {MSM} turunu kullanır. Karma listeler: TourFold'un ilk {KT} "
         f"adayı + başka listeden {KC - KT} aday; birleşim en çok {C3['k_union']} aday.",
         f"- Aşama süreleri (dk, tüm oturumların toplamı): {SUMMARY.get('stage_minutes', {})}", ""]
    st = SUMMARY.get("selftest")
    if st:
        bad = [k for k, v in {**st.get("checks", {}), **st.get("warnings", {})}.items() if not v]
        L += ["## 1. Öz-sınama", "", "Tüm kontroller geçti." if not bad else "Geçmeyen: " + ", ".join(bad), ""]
    tr = SUMMARY.get("training_v2") or []
    if tr:
        h = tr[-1]
        keys = [k for k in h if k.startswith("val_missing_per_1k_M")]
        L += ["Kullanılan model (v2 eğitiminin son epoch'u; doğrulama, eksik tur kenarı / 1000, ilk 5 aday): "
              + ", ".join(f"{k.replace('val_missing_per_1k_', '')} = {h[k]:.3f}" for k in keys)
              + f", α+ascent = {h.get('val_missing_per_1k_alpha', float('nan')):.3f}", ""]
    cov = RESULTS.get("coverage")
    if cov:
        c = pd.DataFrame(cov["coverage"])
        rows_order = [v for v in COV_NAMES.values() if v in set(c["method"])]
        cols = [g for g in cov["groups"] if g in set(c["group"])]

        def piv(v, r):
            return md(c.pivot_table(index="method", columns="group", values=v).reindex(index=rows_order, columns=cols).round(r))
        L += ["## 2. Aday listelerinin kalitesi (LKH'siz, referans turlara göre)", "",
              "TourFold-v2(M) = modelin M turluk toplulukla verdiği sıralama · 3+POPMUSIC 2 = modelin ilk 3 adayı, "
              "ardından POPMUSIC listesinden listede olmayan ilk 2 aday · 3+yakın 2 = aynısı en yakın komşularla · "
              "∪ POPMUSIC = modelin ilk 5 adayı + POPMUSIC'in listede olmayan adayları (en çok 8) · sayım+ = topluluk "
              "sıklığı · POPMUSIC ve α+ascent = LKH-3'ün aday kümeleri (ilk 5). Referans turlar α-8 grafiğinde "
              "bulunduğu için α+ascent satırı biraz iyimserdir.", "",
              "Eksik tur kenarı / 1000 düğüm (listenin tamamı; küçük = iyi):", "", piv("missing_per_1k", 2), "",
              "İlk 2 aday içinde kalan tur kenarı oranı (büyük = iyi):", "", piv("cov2", 4), "",
              "İlk 3 aday içinde kalan tur kenarı oranı (büyük = iyi):", "", piv("cov3", 4), "",
              "Ortalama liste uzunluğu (aday / düğüm):", "", piv("list_len", 2), ""]
        vb = pd.DataFrame(cov["val_by_dist"])
        if len(vb):
            L += ["Doğrulama kümesi, dağılıma göre (eksik tur kenarı / 1000):", "",
                  md(vb.pivot_table(index="method", columns="sub", values="missing_per_1k")
                     .reindex(index=[r for r in rows_order if r in set(vb["method"])]).round(2)), ""]
    lk = RESULTS.get("lkh_eval")
    if lk:
        L += ["## 3. LKH ile süre–kalite v3 (aynı oturum; toplam süre = topluluk + model + LKH)", "",
              "Yapılandırmalar:", "",
              f"- **{CONF['pop']}**: LKH-3'ün kendi POPMUSIC adayları (ascent yok) ve kendi başlangıç turu; taban.",
              f"- **{CONF['ctrl']}**: aynı türden POPMUSIC adayları (topluluk koşusunun ürettiği) + topluluğun en iyi turu "
              "başlangıç olarak; model yok. Topluluğun ve başlangıç turunun katkısını ölçer.",
              f"- **{CONF['cnt']}**: topluluk sıklığına göre sıralanmış komşular (genişletilmiş grafik).",
              f"- **{CONF['tf5']}**: modelin ilk {KC} adayı" + (f" (v2'deki 'TourFold-v2({MS})' ile aynı yapılandırma)."
                                                              if MS == MB else "."),
              f"- **{CONF['tfpop']}**, **{CONF['tfnear']}**, **{CONF['tfuni']}**: karma listeler (bölüm 2).",
              f"- **{CONF['tf4pop']}**: aynı karma liste, topluluğun yalnız ilk {MSM} turuyla (topluluk süresi daha kısa).",
              "",
              f"Kontrol ve rehberli koşular topluluğun en iyi turundan başlar ('{CONF['tf4pop']}': ilk {MSM} turun "
              "en iyisi); LKH bu turun kenarlarını aday kümesine kendisi ekler. Topluluk ve modelin süresi bütçeden düşülür. "
              "En iyi bilinen: TSPLIB'de optimum; diğerlerinde referans tur ile v2 ve v3 koşularının en iyisi.", "",
              "Bütçenin %10, %25, %50 ve %100'ünde en iyi bilinene göre fark (%, ortalama; nan = o ana kadar tur yok):", "",
              md(pd.DataFrame(lk["gap"]).round(4), index=False), "",
              "Hedef farka ulaşma süresi (sn, medyan; inf = bütçe içinde ulaşılamadı):", "",
              md(pd.DataFrame(lk["time_to_gap"]).round(2), index=False), ""]
        if lk.get("offsets"):
            o_ = pd.DataFrame(lk["offsets"])
            L += ["LKH başlamadan önce harcanan süre (sn, ortalama; topluluk + model + liste; bütçeden düşülür):", "",
                  md(o_.pivot_table(index="config", columns="set", values="toplam", sort=False).round(2)), ""]
        fmt_p = lambda v: "nan" if v != v else f"{v:.3g}"   # noqa: E731
        p100 = pd.DataFrame(lk["pooled"].get("gap@100%B", []))
        if len(p100):
            p100["p (işaret testi)"] = p100["p (işaret testi)"].map(fmt_p)
            L += ["Toplu kazanç/beraberlik/kayıp, bütçenin tamamında (K/B/K = yapılandırma rakipten iyi / eşit / kötü; "
                  "boyut sınıfına göre ve tüm örnekler; p: iki yönlü işaret testi, beraberlikler hariç; p değerleri "
                  "çoklu karşılaştırma için düzeltilmemiştir):", "",
                  md(p100, index=False), ""]
            tot = p100[["yapılandırma", "rakip"]].copy()
            for col in ("gap@25%B", "gap@50%B", "gap@100%B"):
                q = pd.DataFrame(lk["pooled"].get(col, []))
                if len(q) == len(tot):
                    tag = f"%{col[4:-2]} bütçe"
                    tot[f"K/B/K {tag}"], tot[f"p {tag}"] = q["tümü"].values, q["p (işaret testi)"].map(fmt_p).values
            L += ["Tüm örnekler, bütçenin %25, %50 ve %100'ünde (K/B/K ve işaret testi p):", "", md(tot, index=False), ""]
        if lk.get("wtl"):
            wt = pd.DataFrame(lk["wtl"])
            wt = wt[wt["config"] == CONF["tfpop"]]
            if len(wt):
                wt["K/B/K"] = wt["wins"].astype(str) + "/" + wt["ties"].astype(str) + "/" + wt["losses"].astype(str)
                L += [f"Örnek kümesine göre '{CONF['tfpop']}' K/B/K (tümü: v3/results/lkh_eval_v3_wtl.csv):", "",
                      md(wt.pivot_table(index=["set", "at"], columns="vs", values="K/B/K", aggfunc="first").reset_index(),
                         index=False), ""]
        if lk.get("repeat_v2"):
            L += ["v2 ile tekrar karşılaştırması (aynı yapılandırma, aynı örnekler, aynı en iyi bilinen; fark yalnız "
                  "zamanlama gürültüsünden gelir):", "", md(pd.DataFrame(lk["repeat_v2"]).round(4), index=False), ""]
            if "ens_time_ratio_v3_v2" in SUMMARY:
                L += [f"Topluluk süresi v3/v2 oranı (medyan): {SUMMARY['ens_time_ratio_v3_v2']:.3f}", ""]
    if SUMMARY.get("errors"):
        L += ["## Hatalar", ""] + [f"- {k}: {v}" for k, v in SUMMARY["errors"].items()] + [""]
    L += ["## Dosyalar", "", "- v3/results: tablolar (csv, json) · v3/figures: grafikler · v3/log_v3.txt: ayrıntılı kayıt", ""]
    text = "\n".join(L)
    with open(f"{R3}/RAPOR_v3.md", "w") as fh:
        fh.write(text)
    jdump(SUMMARY, SUMMARY_PATH)
    return text


def bundle_results_v3(download=True):
    import zipfile
    tmp = f"{LOCAL}/TourFold_v3_sonuclar.zip"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for rel in ("RAPOR_v3.md", "log_v3.txt", "status_v3.json", "config_v3.json"):
            if os.path.exists(f"{R3}/{rel}"):
                z.write(f"{R3}/{rel}", rel)
        for pth, arc in ((CFG_PATH, "config.json"), (C2_PATH, "config_v2.json")):
            if os.path.exists(pth):
                z.write(pth, arc)
        for sub in ("results", "figures"):
            for p_ in sorted(glob.glob(f"{R3}/{sub}/**/*", recursive=True)):
                if os.path.isfile(p_) and not p_.endswith((".tmp", ".part")):
                    z.write(p_, os.path.relpath(p_, R3))
    shutil.copy(tmp, f"{R3}/TourFold_v3_sonuclar.zip")
    msg = f"Sonuç paketi Drive'da: {R3}/TourFold_v3_sonuclar.zip"
    if download and not TEST_MODE:
        import threading
        from google.colab import files
        th = threading.Thread(target=files.download, args=(tmp,), daemon=True)
        th.start()
        th.join(timeout=120)
        msg += " (tarayıcınıza da indiriliyor)"
    log(msg)


# ---------------------------------------------------------------- run everything (v3)
STOP, REPORT, INTERRUPTED, FAILED = None, None, False, False
try:
    LKH = run("1-lkh", stage_lkh)
    run("1-selftest", stage_selftest, LKH)
    SETS = run("3-eval_sets", eval_instances)
    REF = run("3-eval_ref", stage_eval_sets, LKH, SETS)
    SHARDS = run("4-train_data", stage_train_data, LKH)
    log(f"Kayıtlı veri hazır: {sum(len(v) for v in REF.values())} değerlendirme örneği, {len(SHARDS)} eğitim parçası. "
        f"Kaba tahmin: kalan ~{hfmt(sum(EST.values()))} (ölçülerek güncellenir; son tahmin log_v3.txt ve status_v3.json içinde)")
    if DEV.type != "cuda" and not TEST_MODE:
        STOP = ("GPU bulunamadı. Çalışma zamanını GPU'lu seçip (Çalışma zamanı ▸ Çalışma zamanı türünü değiştir ▸ GPU) "
                "hücreyi yeniden çalıştırın.")
    else:
        TF2 = run("6-train_v2", get_tf2, SHARDS)
        VAL = run("6-val", load_val, SHARDS)
        COV = RESULTS["coverage"] = run("7-coverage_v3", stage_coverage_v3, TF2, VAL, REF, critical=False)
        if COV:
            SUMMARY["coverage_v3"] = COV
        LK = RESULTS["lkh_eval"] = run("8-lkh_eval_v3", stage_lkh_eval_v3, LKH, TF2, REF, SETS, critical=False)
        if LK:
            SUMMARY["lkh_eval_v3"] = {k: v for k, v in LK.items() if k not in ("runs", "best")}
except BaseException as _e:
    INTERRUPTED, FAILED = isinstance(_e, KeyboardInterrupt), True
    status("error", "failed", error=repr(_e))
    log(f"DURDU: {_e!r}. Hücreyi yeniden çalıştırmak kaldığı yerden dener; hata sürerse {R3}/log_v3.txt ve RAPOR_v3.md "
        "dosyalarını bana gönderin.")
    raise
finally:
    try:
        REPORT = write_report_v3()
    except Exception as _e2:  # noqa: BLE001
        log(f"rapor yazılamadı: {_e2!r}")
    if FAILED:
        try:
            bundle_results_v3(download=not INTERRUPTED)
        except Exception as _e3:  # noqa: BLE001
            log(f"sonuç paketi hazırlanamadı: {_e3!r}")
if STOP:
    status("waiting_for_gpu", "paused")
    log(STOP)
elif SUMMARY["errors"]:
    status("done_with_errors", "finished", errors=list(SUMMARY["errors"]))
    log(f"Bitti, ancak bazı adımlar hata verdi (RAPOR_v3.md > Hatalar). {R3}/RAPOR_v3.md ve log_v3.txt dosyalarını bana gönderin.")
else:
    status("done", "finished")
    log(f"BİTTİ. Rapor: {R3}/RAPOR_v3.md — bu dosyayı (ya da TourFold_v3_sonuclar.zip'i) bana gönderin.")
try:
    bundle_results_v3(download=not STOP)
except Exception as _e3:  # noqa: BLE001
    log(f"sonuç paketi hazırlanamadı: {_e3!r}")
print("\n" + (REPORT or ""))
